# When LLM Judges Need A Decision Tree, Not A Rubric

*Build the same compliance criterion twice in DeepEval, once with G-Eval and once with a deterministic DAG metric.*

**Read this if**

- You need a metric for a rule your organization wrote, which no library ships.
- Your criterion mixes a hard rule with a soft judgement, and one score cannot carry both.
- An auditor has asked why the same output scored differently twice.

The previous post left Halcyon with a Docs Assistant, and with its search step scored apart from its answer-writing step. One metric saw both traps, and only because it held the reference answer. What Halcyon still cannot do is check a live answer against a rule of its own. Its one house rule is a single sentence. The model that scores it rewrites its own checklist for that rule on every construction.

Halcyon is a payments API vendor with about two hundred staff. Its customers are developers integrating card payments. The Docs Assistant answers their questions from the public documentation. A search step, the retriever, finds a few slices of pages, called chunks. A language model, the generator, writes an answer from them. The developer writes code against the answer. The support triage assistant summarizes inbound tickets, and a severity step routes each summary. Nobody at Halcyon reviews either output before it acts. Code ships, or a ticket waits four hours instead of one.

In March a developer shipped the webhook recipe the assistant gave her. It was the deprecated one, and every event failed. The docs team wrote a rule that afternoon. The assistant must never hand a developer a deprecated scheme without saying so. Halcyon's compliance lead asked for that rule as a check on every answer, with one condition. The score has to be the same tomorrow as today. The team wrote the rule into a metric as one sentence. They scored the incident answer twice and got two different numbers.

## One rule, two shapes

A judge is a second language model that reads an output and returns a score with a reason. A metric is the code around the judge. It decides what the judge is asked, how many times, and how the reply becomes a number. A test case is the record a metric reads. For the Docs Assistant that is the question, the answer and the chunks the retriever returned. Sometimes it also holds a reference answer we wrote by hand. A metric that needs the reference is reference-based. One that reads only what the system produced is referenceless. Only referenceless metrics can run on live traffic. The threshold is the minimum score that counts as a pass. Every DeepEval score reads higher-is-better, and the threshold is a floor.

DeepEval ships two ways to turn a rule of our own into a metric. The first is G-Eval, which takes the rule as one paragraph. The judge writes a marking scheme from that paragraph. Then it follows the scheme and returns a score from 0 to 10. The second is a DAG, a directed acyclic graph. That is a decision tree where we write every question. Each question is a node. The judge answers one node at a time, and each answer picks a branch. Some branches end at a score we chose. Others continue to the next question.

A code review makes the difference concrete. Asking a reviewer "is this pull request good?" is G-Eval. The reviewer decides what to check. A CI pipeline is a DAG. Lint runs before the tests, and the tests run before anyone reads the diff. A red gate is red regardless of how the diff reads. The analogy stops at the gates themselves. A CI gate is a program with one answer. A DAG gate is still a judge call, and a judge can misread a gate. What the tree fixes is the path from answers to score, never the answers. DeepEval's own documentation says the same in one line.

The arithmetic of the two shapes is short. G-Eval costs two judge calls per test case, one to write the steps and one to score. If we pass the steps in ourselves it costs one. A DAG costs one judge call per node it visits. A path that ends at a fixed score costs one more, to write a reason. The tree we build here visits three nodes at most.

Here is the plan. We start with the metric the series already has, a one-sentence G-Eval for the P1 rule. That is Halcyon's clause that any ticket reporting affected funds gets top priority. We watch its marking scheme move between constructions. Then we write Halcyon's developer-answer rule twice, once as G-Eval and once as a tree. We score the same ten answers five times with each. We report the spread per question, the highest score minus the lowest across runs. We report the verdicts against a key we labelled by hand. Then the payoff. The deprecated recipe that scored 1.00 on faithfulness meets a metric that knows what deprecated means. Faithfulness only asks whether an answer contradicts the chunks it was given. Then the same rule at two values of top-k, the number of chunks the retriever hands the generator. Then the P1 rule as a tree, briefly.

The notebook reads `GEMINI_API_KEY` from the environment. A full run costs about a dollar, and the exact bill is printed at the end.

In [1]:
# Shared Halcyon scaffolding, built in post 0 and re-created verbatim in every later post.
# !uv pip install deepeval==4.2.1 langgraph==1.2.11 langchain-core==1.6.2 langchain-google-genai==4.4.0 google-genai==2.22.0
import os, re, json, time, hashlib, logging, statistics
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from typing import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from deepeval.dataset import Golden

logging.getLogger("google_genai.models").setLevel(logging.ERROR)   # silence an SDK advisory about tool calling

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("Set GEMINI_API_KEY in the environment before running this notebook.")

JUDGE_MODEL = "gemini-3.6-flash"      # the judge every later post uses; this post never calls it
APP_MODEL = "gemini-3.6-flash"        # the model inside the assistant under test
MAX_CONCURRENCY = 4                   # model calls in flight at any moment
PRICE_IN, PRICE_OUT = 0.75 / 1e6, 3.75 / 1e6   # USD per token, gemini-3.6-flash standard tier, 2026 rate
T0 = time.time()
USAGE = []                            # one usage_metadata dict per model call, for the cost ledger


def with_retry(fn, *args, attempts=5, base_delay=2.0):
    """Call fn(*args); on a 429 or 503 wait 2, 4, 8, 16 seconds and try again."""
    for attempt in range(attempts):
        try:
            return fn(*args)
        except Exception as exc:
            transient = any(code in str(exc) for code in ("429", "503", "RESOURCE_EXHAUSTED"))
            if not transient or attempt == attempts - 1:
                raise
            time.sleep(base_delay * 2 ** attempt)


# No temperature, top_p or top_k: Google lists them as deprecated sampling parameters to strip from requests.
llm = ChatGoogleGenerativeAI(model=APP_MODEL, google_api_key=os.environ["GEMINI_API_KEY"], max_retries=0)

DOCS = [dict(zip(('slug', 'title', 'status', 'updated', 'body'), row)) for row in [
    ("quickstart", "Quickstart", "current", "2026-03-01", "Create an account, then generate a test API key from the dashboard; test keys start with sk_test_ and live keys with sk_live_. Install the SDK with pip install halcyon (the current SDK is 3.x). Create your first charge with POST /v1/charges, passing amount in minor units (an integer: 1999 means 19.99), a lowercase ISO 4217 currency such as usd or eur, and a source token from Halcyon.js. Send the Halcyon-Version header with the API version you tested against so later changes do not reach you until you opt in. Use test card 4242 4242 4242 4242 for a successful payment and 4000 0000 0000 0002 for a decline."),
    ("api-conventions", "API conventions", "current", "2026-03-01", "The API is served over HTTPS at https://api.halcyon.dev and is date-versioned: send Halcyon-Version: 2026-03-01 (or an earlier version) with each request; requests without the header use the account's default version. Amounts are integers in minor units and currencies are lowercase ISO 4217 codes. Every POST to a money-moving endpoint (charges, refunds, payouts) must include an Idempotency-Key header, a client-generated unique string; a UUID v4 is recommended. Replaying a request with the same key within 24 hours returns the original response instead of moving money again. Requests without the header are accepted for backward compatibility, so a retried POST without one creates a second charge. Keys are scoped to the account and to the endpoint."),
    ("authentication", "Authentication", "current", "2026-01-15", "Authenticate every request with an Authorization: Bearer header carrying your secret key. Test-mode keys (sk_test_) never move real money; live-mode keys (sk_live_) do. Restricted keys can be limited to read-only access or to specific resources. Rotate a key from the dashboard: the new key is active immediately and the old key keeps working for 24 hours so deployments can roll over without downtime; revoke it sooner from the same screen if it was exposed. A missing or invalid key returns HTTP 401 with type authentication_error. Never send secret keys from browsers or mobile apps; use Halcyon.js publishable keys (pk_) there."),
    ("errors", "Errors", "current", "2026-03-01", "Errors return a JSON body with type, code, message and request_id. Types map to HTTP status: invalid_request_error (400), authentication_error (401), card_error (402), rate_limit_error (429) and api_error (500 to 504). card_error codes include card_declined, insufficient_funds, expired_card and incorrect_cvc; show the cardholder a generic message and let them try another card. Requests that return 5xx may have partially completed: retry them with exponential backoff, and quote the request_id when you contact support. Do not retry 4xx errors other than 429."),
    ("rate-limits", "Rate limits", "current", "2025-11-01", "Live mode allows 100 requests per second per account with short bursts to 200; test mode allows 25 requests per second. Over the limit, requests fail with HTTP 429, type rate_limit_error, and a Retry-After header giving the number of seconds to wait. Back off exponentially with jitter rather than retrying in a tight loop, and spread bulk work such as migrations or invoice runs over time or through a queue. Webhook deliveries to your endpoint do not count against the limit. Contact support with your expected peak if you need a higher limit."),
    ("charges-create", "Create a charge", "current", "2026-03-01", "POST /v1/charges creates a charge. Required parameters: amount (integer, minor units), currency (lowercase ISO 4217) and source (a token from Halcyon.js or a saved payment method id starting pm_). Optional: description, metadata (up to 20 key-value pairs), and capture, which defaults to true; pass capture=false to authorize only, then capture within 7 days with POST /v1/charges/{id}/capture. The response is a charge object with id (ch_), status (succeeded, pending or failed), amount, currency, amount_refunded and failure_code. Card declines return HTTP 402 with a card_error body; a charge that could not be attempted at all returns 400."),
    ("refunds", "Refunds", "current", "2026-02-10", "POST /v1/refunds with charge (a ch_ id) refunds a charge in full; pass amount to refund part of it. A charge can be refunded several times until the refunded total reaches the original amount; a request beyond the remaining balance returns 400 with code amount_exceeds_refundable. Refunds of a pending charge are rejected until the charge succeeds. The refund object has id (re_), status (pending, succeeded or failed) and amount. Refunds settle to the cardholder in 5 to 10 business days depending on the issuing bank; status succeeded means Halcyon has released the funds, not that the cardholder has seen them yet."),
    ("webhooks-overview", "Webhooks", "current", "2026-03-01", "Halcyon notifies your endpoint about events by POSTing a JSON event object with id (evt_), type, created and data. Event types include charge.succeeded, charge.failed, refund.succeeded and dispute.opened. Your endpoint must return a 2xx within 10 seconds; otherwise the delivery is retried with exponential backoff (1 minute, 5 minutes, 30 minutes, 2 hours, then every 6 hours) for up to 72 hours, after which the event is marked failed and shown in the dashboard. Delivery is at-least-once: the same event can arrive more than once and events can arrive out of order, so store the event id and ignore repeats. Always verify the signature header before trusting an event; see the webhook signatures pages."),
    ("webhook-signatures-v1", "Webhook signatures (v1)", "deprecated", "2026-03-01", "Each webhook request carries an X-Halcyon-Signature header containing a hex-encoded HMAC-SHA1 of the raw request body, keyed with your endpoint's signing secret (whsec_). To verify, compute HMAC-SHA1 over the exact bytes you received, before any JSON parsing, and compare with the header using a constant-time comparison. Reject the event if the values differ. Do not re-serialize the JSON before hashing, since key ordering and whitespace change the digest. This scheme was deprecated on 2026-03-01 in favour of webhook-signatures-v2, which uses HMAC-SHA256 with a timestamped Halcyon-Signature header; it remains in service for accounts pinned to API versions before 2026-03-01."),
    ("webhook-signatures-v2", "Webhook signatures (v2)", "current", "2026-03-01", "Each webhook request carries a Halcyon-Signature header of the form t=<unix timestamp>,v2=<hex digest>. The signed payload is the timestamp, a period, and the raw request body. To verify, compute HMAC-SHA256 of that payload with your endpoint's signing secret (whsec_), compare with the v2 value using a constant-time comparison, and reject the event if the timestamp is more than 300 seconds old, which defeats replay of captured requests. During a signing secret rotation the header carries two v2 values for 24 hours; accept the event if either matches. This is the default scheme for API version 2026-03-01 and later, and the only scheme SDK 3.x verifies."),
    ("changelog-2026-03", "Changelog: API version 2026-03-01", "current", "2026-03-01", "API version 2026-03-01. Webhook signatures v2 (HMAC-SHA256, timestamped Halcyon-Signature header) are now the default; v1 (HMAC-SHA1, X-Halcyon-Signature) is deprecated and is sent only to accounts pinned to earlier versions. Halcyon SDK 3.0 ships with v2 verification helpers and no longer verifies v1 headers; SDK 2.x continues to verify v1 only. The Idempotency-Key replay window is extended from 1 hour to 24 hours, and reusing a key with a different request body now returns 400 with code idempotency_key_reused instead of silently returning the earlier response. Request and response shapes for charges and refunds are unchanged."),
    ("support-sla", "Support and SLA", "current", "2026-01-15", "Support is available to all accounts through the dashboard and support@halcyon.dev. Tickets are triaged into three priorities. P1: any report that funds have been affected, including duplicate or double charges, charges of the wrong amount, and refunds or payouts that did not arrive, regardless of how the report is worded or whether the ticket is mainly about something else; first response within 1 hour, 24 hours a day. P2: a live-mode integration that is failing, such as authentication errors, webhook deliveries or signature verification failing, or unexpected 4xx or 5xx responses from live endpoints; first response within 4 business hours. P3: how-to questions, documentation gaps, feature requests and anything in test mode; first response within 1 business day."),
]]

TICKETS = [dict(zip(('id', 'submitted', 'subject', 'body'), row)) for row in [
    ("T-1001", "2026-08-03", "Which signature header am I supposed to verify?", "We're integrating webhooks and every event we receive has an X-Halcyon-Signature header but no Halcyon-Signature header, so the verify() helper in SDK 3.1 throws 'no v2 signature present'. The webhook overview says to verify signatures and links to two pages. Our account was created in 2025 and I think we're pinned to 2025-11-01. Is v1 still supported, do we need to move to v2, and how do we change the API version without breaking our existing charge flow? This is blocking our launch on the live account."),
    ("T-1002", "2026-08-04", "429s in test mode when our integration suite runs", "Our CI runs about 40 requests per second against test mode at peak and we get bursts of rate_limit_error with Retry-After: 1. Live mode is fine. Is the test-mode limit documented anywhere, and can it be raised for CI? We can add a throttle but it makes the suite roughly three times slower."),
    ("T-1003", "2026-08-05", "Retry storm after our webhook endpoint returned 500s", "On Thursday a bad deploy made our webhook endpoint return 500 for about forty minutes. Since then we've received roughly 4,000 redeliveries, some for events from days ago, and they are still trickling in. Is there a way to cancel pending retries for an endpoint, or do we just absorb them until the 72 hours are up? Probably unrelated, but while going through the logs we found a few orders from that afternoon that were charged twice, two charge ids each; we assume our order service retried the charge call on timeout and we'll refund them on our side. Mainly we need the retries to stop."),
    ("T-1004", "2026-08-06", "Getting 400 idempotency_key_reused on retries", "Since we moved to API version 2026-03-01 last week, our retry path fails with 400 idempotency_key_reused. We generate one Idempotency-Key per order and reuse it if the first attempt fails, which used to work. Looking closer, our retry adds a metadata.attempt field to the body. Is the body compared exactly? What is the recommended pattern for retries where the body legitimately changes? This is live traffic; failed retries currently land in a dead-letter queue and someone replays them by hand."),
    ("T-1005", "2026-08-07", "Partial refund returns amount_exceeds_refundable", "A customer paid 120.00 EUR and we refunded 80.00 EUR last week. Refunding the remaining 40.00 EUR today returns 400 amount_exceeds_refundable. It turns out our cancellation flow had already issued a separate 40.00 EUR refund that nobody noticed, so the error is correct. What we actually need: is there a field on the charge that tells us the remaining refundable balance, so we can check before calling refunds? And is there any per-month cap on refunds?"),
    ("T-1006", "2026-08-10", "SDK 3.0 upgrade broke webhook verification, rolled back", "We upgraded from SDK 2.9 to 3.0 on Monday. From the first deploy every webhook failed verification with 'no v2 signature present' and our fulfilment queue stalled. We rolled back to 2.9 after about 25 minutes and verification works again. Our account is on API version 2025-11-01. What exactly changed in 3.0, and what is the migration order: SDK first or API version first? One side effect: while the queue was stalled a colleague re-ran the stuck jobs by hand and two customers were charged a second time; we've refunded one and are chasing the other. The verification question is the one we need answered before we try again."),
    ("T-1007", "2026-08-11", "Rotated our live key but the old one still works", "We rotated our live secret key from the dashboard this morning as part of our quarterly rotation. Two hours later, requests using the old key still succeed. Is rotation delayed, or did it not take? We expected the old key to stop working straight away. Also, is there an audit log showing which key made which request?"),
    ("T-1008", "2026-08-12", "How do we simulate a dispute in test mode?", "We're building our dispute.opened handler and can't find a way to trigger a dispute against a test-mode charge. Is there a test card number or a dashboard button that opens a dispute? Also, what fields does the dispute object carry? Not urgent, we're a few weeks from launch."),
    ("T-1009", "2026-08-13", "Batch import script times out around row 200", "We're migrating 1,800 customers from our old provider. The script imports each saved payment method and immediately creates the first invoice charge. It runs fine for about 200 rows and then requests start timing out at 30 seconds; the script retries each failed row up to three times and eventually finishes, but the whole run takes hours. Is there a batch endpoint, or a recommended concurrency for imports? Somewhat related: our finance team says a dozen or so of the imported customers have two invoice charges instead of one, which we think is the retry path. We can clean that up ourselves once the import is stable."),
    ("T-1010", "2026-08-14", "Duplicate webhook deliveries and events arriving out of order", "We're seeing the same evt_ id delivered two or three times, sometimes minutes apart, and occasionally a refund.succeeded arrives before the charge.succeeded for the same order. No money problem that we can see, every charge is correct in the dashboard, but our handler assumed one delivery per event and now double-writes some rows. Is this expected behaviour? Should we be deduplicating on our side and, if so, on which field?"),
    ("T-1011", "2026-08-17", "Intermittent 502s from POST /v1/charges since Tuesday", "Roughly one in fifty POST /v1/charges calls returns a 502 with an HTML body rather than your JSON error format, all from the eu-west region. Our client retries 5xx with backoff as your errors page recommends and the retry almost always succeeds, so checkout is mostly working. Request ids for six failures are attached. Two customers have emailed our support saying their statement shows the same order twice, which I assume is the retry; we'll sort that out with them. The 502s are what we need you to look at."),
    ("T-1012", "2026-08-18", "Do you support Apple Pay and Google Pay?", "Our product team wants wallet payments at checkout. I can't find Apple Pay or Google Pay anywhere in the docs. Is it supported through Halcyon.js, on a roadmap, or would we need a second provider for wallets? Also, is there a fee difference for wallet payments?"),
]]

def corpus_fingerprint(docs, tickets):
    return hashlib.sha256(json.dumps([docs, tickets], sort_keys=True).encode()).hexdigest()[:12]

SUMMARIZE = ("You write one-line summaries for Halcyon's support triage queue. Summarize the ticket below "
             "in one sentence of at most 25 words, in plain English.\n\nTicket:\n{ticket}")

def ticket_text(ticket):
    return f"Subject: {ticket['subject']}\n\n{ticket['body']}"

class AppState(TypedDict):
    question: str
    context: list[str] | None
    answer: str

SYSTEM = ("You are Halcyon's developer support assistant. Halcyon is a payments API vendor. When "
          "documentation pages are supplied, answer from them alone and say so if they do not cover the "
          "question. Be concise and specific.")

def respond(state: AppState) -> dict:
    prompt = state["question"]
    if state.get("context"):
        prompt = "Documentation pages:\n\n" + "\n\n".join(state["context"]) + f"\n\nQuestion: {prompt}"
    reply = with_retry(llm.invoke, [SystemMessage(SYSTEM), HumanMessage(prompt)])
    USAGE.append(reply.usage_metadata or {})
    return {"answer": reply.text.strip()}

builder = StateGraph(AppState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)
APP = builder.compile()

def answer(question, context=None):
    return APP.invoke({"question": question, "context": context})["answer"]

SLA = next(d for d in DOCS if d["slug"] == "support-sla")["body"]
CLASSIFY = ("Assign a support priority to the ticket summary below using Halcyon's SLA. Reply with exactly "
            "one of P1, P2 or P3 and nothing else.\n\nSLA:\n{sla}\n\nTicket summary: {summary}")

def triage(ticket):
    """Ticket in, one-sentence summary plus severity out. The severity step sees only the summary."""
    summary = answer(SUMMARIZE.format(ticket=ticket_text(ticket)))
    verdict = re.search(r"P[123]", answer(CLASSIFY.format(sla=SLA, summary=summary)))
    return {"id": ticket["id"], "summary": summary, "severity": verdict.group(0) if verdict else "unparsed"}

GOLDEN_SPEC = {  # severity, trap, pages, reference summary, why it exists
    "T-1001": ("P2", "two-truths", ["webhook-signatures-v1", "webhook-signatures-v2", "changelog-2026-03", "api-conventions"], "Live webhooks carry only the v1 X-Halcyon-Signature header, so SDK 3.1 verification fails; asks whether to move the account to API version 2026-03-01.", "Both signature pages are published and both are correct for someone; the right answer depends on the account's API version."),
    "T-1002": ("P3", None, ["rate-limits"], "CI hits test-mode rate limits at 40 requests per second; asks whether the limit is documented or can be raised.", "A plain how-to with one page that answers it fully; the control case."),
    "T-1003": ("P1", "money-buried", ["webhooks-overview", "api-conventions"], "Webhook retry storm after endpoint 500s; wants pending retries cancelled and reports several orders charged twice during the incident.", "The ticket is about retries; the double charge is an aside at the end. P1 under the SLA."),
    "T-1004": ("P2", None, ["api-conventions", "changelog-2026-03", "errors"], "Retries reusing an Idempotency-Key fail with 400 idempotency_key_reused on live traffic because the retry body differs; asks for the correct retry pattern.", "Needs two pages combined: the conventions rule and the changelog entry that changed it."),
    "T-1005": ("P3", None, ["refunds", "charges-create"], "Second partial refund rejected with amount_exceeds_refundable; asks how to read the remaining refundable balance before refunding.", "Mentions refunds and amounts without any funds being affected; a P3 that sounds like money."),
    "T-1006": ("P1", "money-buried", ["changelog-2026-03", "webhook-signatures-v2", "webhook-signatures-v1"], "SDK 3.0 upgrade made webhook verification fail and was rolled back; two customers were charged twice during the stall; asks the migration order.", "Leads with a failed upgrade; the double charge is a side effect the developer plays down."),
    "T-1007": ("P3", None, ["authentication"], "After a scheduled key rotation the old live key still works two hours later; asks whether that is expected and for how long.", "Sounds like a security incident and is documented behaviour; tests calm reading of docs."),
    "T-1008": ("P3", "no-coverage", [], "Cannot trigger a dispute against a test-mode charge; asks how to simulate one and what the dispute object contains.", "No page covers this. The correct answer is to say so, not to invent a test card."),
    "T-1009": ("P1", "money-buried", ["rate-limits", "api-conventions", "charges-create"], "Bulk import of 1,800 customers times out after about 200 rows; retries have created duplicate invoice charges for a dozen customers.", "Leads with timeouts and batch advice; the duplicate charges are 'somewhat related'."),
    "T-1010": ("P3", "near-miss", ["webhooks-overview"], "Same webhook event delivered several times and out of order; no money affected; asks whether that is expected and how to deduplicate.", "Uses 'duplicate' and 'double' about deliveries, not charges. A keyword rule fires; the SLA does not."),
    "T-1011": ("P1", "money-buried", ["errors", "charges-create", "api-conventions"], "About one in fifty live POST /v1/charges calls returns 502 from eu-west; retries have charged two customers twice.", "Leads with 502s and follows the errors page's own retry advice; the retry is what double-charged customers."),
    "T-1012": ("P3", "no-coverage", [], "Asks whether Apple Pay and Google Pay are supported through Halcyon.js or planned.", "No page covers wallets. Tests whether the assistant admits a documentation gap."),
}

GOLDENS = []
for t in TICKETS:
    severity, trap, pages, summary, reason = GOLDEN_SPEC[t["id"]]
    GOLDENS.append(Golden(name=t["id"], input=ticket_text(t), expected_output=summary,
                          additional_metadata={"expected_severity": severity, "trap": trap,
                                               "pages": pages, "reason": reason}))

CORPUS_FINGERPRINT = corpus_fingerprint(DOCS, TICKETS)
assert CORPUS_FINGERPRINT == "e4485ba0f187", "DOCS or TICKETS drifted from post 0"
print("corpus fingerprint", CORPUS_FINGERPRINT)

corpus fingerprint e4485ba0f187


That is the shared setup cell from the first post in the series, unchanged. The fingerprint, a hash over the pages and tickets, matched. The next five cells are carried forward too. First the judge and ledger cell from the earlier metric posts. Then four cells from the retrieval post. The retry wrapper and the Gemini embedder. The chunked index. The ten developer questions with their own fingerprint. And the retrieve-then-generate graph that answers them at top-k 3. The retriever is the retrieval post's inline one, unchanged. We do not rebuild it here.

In [2]:
# Carried forward from the earlier metric posts. Everything above is the shared setup cell, unchanged.
import asyncio
import google.genai.models as genai_models
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, SummarizationMetric
from deepeval.models import GeminiModel
from deepeval.models.llms.constants import GEMINI_MODELS_DATA
from deepeval.test_case import LLMTestCase

MONEY = re.compile(r"twice|double|duplicat|second (charge|time)|two (charge|invoice)|charged again|"
                   r"extra charge|multiple charge|overcharg|billed (again|twice)", re.I)   # the trap check
MONEY_TICKET_IDS = ("T-1003", "T-1006", "T-1009", "T-1011")

JUDGE_USAGE = []   # (usage_metadata, temperature sent) for every judge call; DeepEval's counter omits thinking
_generate_content = genai_models.AsyncModels.generate_content

async def counted_generate_content(self, *args, **kwargs):
    response = await _generate_content(self, *args, **kwargs)
    JUDGE_USAGE.append((response.usage_metadata, kwargs["config"].temperature))
    return response

genai_models.AsyncModels.generate_content = counted_generate_content

def judge_calls():
    return len(JUDGE_USAGE)

registry_price = GEMINI_MODELS_DATA[JUDGE_MODEL].input_price, GEMINI_MODELS_DATA[JUDGE_MODEL].output_price
# Same key variable as the app model: GeminiModel would otherwise look for GOOGLE_API_KEY.
judge = GeminiModel(model=JUDGE_MODEL, api_key=os.environ["GEMINI_API_KEY"],
                    cost_per_input_token=PRICE_IN, cost_per_output_token=PRICE_OUT)

async def _measure(metric, case, gate):
    async with gate:
        await metric.a_measure(case, _show_indicator=False)   # no progress spinner in a notebook
    return metric

async def score(metric_cls, cases, **kwargs):
    """One fresh metric per case, MAX_CONCURRENCY cases in flight; returns {case name: measured metric}."""
    gate = asyncio.Semaphore(MAX_CONCURRENCY)
    metrics = [metric_cls(model=judge, **kwargs) for _ in cases]
    await asyncio.gather(*(_measure(m, c, gate) for m, c in zip(metrics, cases)))
    return {c.name: m for c, m in zip(cases, metrics)}

print(f"deepeval {version('deepeval')}   judge {judge.get_model_name()}")
print(f"temperature DeepEval will send: {judge.temperature}   "
      f"(its registry says supports_temperature={judge.supports_temperature()})")
print(f"price per 1M tokens we set: ${PRICE_IN * 1e6:.2f} in, ${PRICE_OUT * 1e6:.2f} out   "
      f"registry default: ${registry_price[0] * 1e6:.2f} in, ${registry_price[1] * 1e6:.2f} out")
print(f"app model calls: {len(USAGE)}   judge calls: {judge_calls()}")

deepeval 4.2.1   judge gemini-3.6-flash (Gemini)
temperature DeepEval will send: 0.0   (its registry says supports_temperature=True)
price per 1M tokens we set: $0.75 in, $3.75 out   registry default: $1.50 in, $7.50 out
app model calls: 0   judge calls: 0


In [3]:
# Carried forward from the retrieval post, where this cell was new: the retry wrapper, the Gemini embedder,
# and the chunk and top-k constants. The two cells above are the setup cell and the judge and ledger cell, unchanged.
# !uv pip install chromadb==1.5.9
import numpy as np
import chromadb
from google import genai
from google.genai import types
from google.genai import errors as genai_errors
from deepeval.metrics import ContextualPrecisionMetric, ContextualRecallMetric, ContextualRelevancyMetric
from deepeval.models import DeepEvalBaseEmbeddingModel
import deepeval.models as deepeval_models

EMBED_MODEL = "gemini-embedding-001"
PRICE_EMBED = 0.15 / 1e6           # USD per input token, standard tier; embeddings have no output tokens
CHUNK_WORDS = 60                   # a chunk holds whole sentences, up to this many words
TOP_K = 3                          # chunks the retriever hands to the generator

# The previous post's change to the ledger wrapper: judge calls retry on a 5xx from Google, and each
# failed attempt is recorded so the ledger can report it.
JUDGE_RETRIES = []

async def retrying_generate_content(self, *args, **kwargs):
    for attempt in range(5):
        try:
            response = await _generate_content(self, *args, **kwargs)   # the original client method, saved above
            break
        except genai_errors.ServerError as exc:
            JUDGE_RETRIES.append(type(exc).__name__)
            if attempt == 4:
                raise
            await asyncio.sleep(2.0 * 2 ** attempt)
    JUDGE_USAGE.append((response.usage_metadata, kwargs["config"].temperature))
    return response

genai_models.AsyncModels.generate_content = retrying_generate_content

# Every embedding call goes through embed(), so the ledger sees all of them.
EMBED_CLIENT = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
EMBED_USAGE = []   # one dict per embed_content call: texts sent and Google's token count for them

def embed(texts, task_type):
    """One embed_content call. Returns one vector per text, in the order the texts were given."""
    config = types.EmbedContentConfig(task_type=task_type)
    response = EMBED_CLIENT.models.embed_content(model=EMBED_MODEL, contents=texts, config=config)
    counted = EMBED_CLIENT.models.count_tokens(model=EMBED_MODEL, contents=texts)   # embed responses carry no usage
    EMBED_USAGE.append({"texts": len(texts), "tokens": counted.total_tokens})
    vectors = []
    for embedding in response.embeddings:
        vectors.append(embedding.values)
    return vectors

class GeminiEmbedder(DeepEvalBaseEmbeddingModel):
    """gemini-embedding-001 behind DeepEval's embedding interface. embed_texts is for documents, embed_text for a query."""
    def __init__(self):
        super().__init__(EMBED_MODEL)

    def load_model(self):
        return EMBED_CLIENT

    def embed_texts(self, texts):
        return with_retry(embed, texts, "RETRIEVAL_DOCUMENT")

    def embed_text(self, text):
        return with_retry(embed, [text], "RETRIEVAL_QUERY")[0]

    async def a_embed_texts(self, texts):
        return self.embed_texts(texts)

    async def a_embed_text(self, text):
        return self.embed_text(text)

    def get_model_name(self):
        return EMBED_MODEL

embedder = GeminiEmbedder()

shipped = []
for name in dir(deepeval_models):
    if name.endswith("EmbeddingModel") and name != "DeepEvalBaseEmbeddingModel":
        shipped.append(name)
print(f"embedding classes deepeval {version('deepeval')} ships: {', '.join(shipped)}")
print(f"ours: {embedder.get_model_name()} inside {type(embedder).__name__}, a subclass of DeepEvalBaseEmbeddingModel")
print(f"judge wrapper now: {genai_models.AsyncModels.generate_content.__name__}, up to 5 attempts per call")
print(f"chunk budget {CHUNK_WORDS} words, top-k {TOP_K}   app model calls: {len(USAGE)}   judge calls: {judge_calls()}   embed calls: {len(EMBED_USAGE)}")


embedding classes deepeval 4.2.1 ships: AzureOpenAIEmbeddingModel, LocalEmbeddingModel, OllamaEmbeddingModel, OpenAIEmbeddingModel
ours: gemini-embedding-001 inside GeminiEmbedder, a subclass of DeepEvalBaseEmbeddingModel
judge wrapper now: retrying_generate_content, up to 5 attempts per call
chunk budget 60 words, top-k 3   app model calls: 0   judge calls: 0   embed calls: 0


In [4]:
# Carried forward from the retrieval post: the sentence splitter, the chunker and the embedded index, unchanged.
def sentences(text):
    return [s for s in re.split(r"(?<=[.?!])\s+", text) if s]

def chunk_page(doc):
    """Whole sentences packed into chunks of at most CHUNK_WORDS words. A sentence is never split."""
    chunks = []
    current = []
    for sentence in sentences(doc["body"]):
        words_so_far = len(" ".join(current).split())
        if current and words_so_far + len(sentence.split()) > CHUNK_WORDS:
            chunks.append(" ".join(current))
            current = []
        current.append(sentence)
    if current:
        chunks.append(" ".join(current))
    return chunks

CHUNKS = []   # one record per chunk: part, slug, status, updated, text, and after indexing, vector
for doc in DOCS:
    parts = chunk_page(doc)
    for number, text in enumerate(parts, 1):
        CHUNKS.append({"part": f"{doc['slug']}#{number}", "slug": doc["slug"], "status": doc["status"],
                       "updated": doc["updated"], "text": text})

texts = []
for chunk in CHUNKS:
    texts.append(chunk["text"])
t0 = time.time()
vectors = embedder.embed_texts(texts)
for chunk, vector in zip(CHUNKS, vectors):
    chunk["vector"] = vector
print(f"{len(CHUNKS)} chunks from {len(DOCS)} pages, embedded in {len(EMBED_USAGE)} call, "
      f"{len(vectors[0])} dimensions each, {time.time() - t0:.1f} s\n")

print(f"{'chunk':<26}{'status':<12}{'words':>5}")
for chunk in CHUNKS:
    print(f"{chunk['part']:<26}{chunk['status']:<12}{len(chunk['text'].split()):>5}")

# The trap check. The chunk that carries the SHA1 instructions must not carry the deprecation notice.
sha1_chunk = None
for chunk in CHUNKS:
    if chunk["slug"] == "webhook-signatures-v1" and "HMAC-SHA1" in chunk["text"]:
        sha1_chunk = chunk
deprecation_in_sha1_chunk = "deprecated" in sha1_chunk["text"]
print(f"\n{sha1_chunk['part']}, status {sha1_chunk['status']}, updated {sha1_chunk['updated']}:")
print(sha1_chunk["text"])
print(f"\ndeprecation notice inside this chunk: {deprecation_in_sha1_chunk}")
assert not deprecation_in_sha1_chunk, "the chunker did not separate the instructions from the deprecation notice"


30 chunks from 12 pages, embedded in 1 call, 3072 dimensions each, 0.7 s

chunk                     status      words
quickstart#1              current        34
quickstart#2              current        55
quickstart#3              current        19
api-conventions#1         current        43
api-conventions#2         current        42
api-conventions#3         current        29
authentication#1          current        35
authentication#2          current        50
authentication#3          current        15
errors#1                  current        49
errors#2                  current        29
rate-limits#1             current        42
rate-limits#2             current        51
charges-create#1          current        56
charges-create#2          current        39
refunds#1                 current        56
refunds#2                 current        45
webhooks-overview#1       current        27
webhooks-overview#2       current        46
webhooks-overview#3       current        40
we

In [5]:
# Carried forward from the retrieval post: DOC_QUESTIONS and their fingerprint, unchanged.
# Ten developer questions with reference answers. The pages field names where the answer lives in DOCS.
DOC_QUESTIONS = [
    {"id": "Q-01", "trap": "deprecated-page", "pages": ["webhook-signatures-v2"],
     "question": "How do I verify the signature on a webhook Halcyon sends me?",
     "expected": "Read the Halcyon-Signature header, which has the form t=<unix timestamp>,v2=<hex digest>. Compute HMAC-SHA256 over the timestamp, a period and the raw request body with the endpoint's signing secret (whsec_), compare it with the v2 value using a constant-time comparison, and reject the event if the timestamp is more than 300 seconds old."},
    {"id": "Q-02", "trap": "missing-requirement", "pages": ["charges-create", "api-conventions"],
     "question": "How do I create a charge?",
     "expected": "POST /v1/charges with amount in minor units, a lowercase ISO 4217 currency and a source token from Halcyon.js or a saved payment method id starting pm_. Because charges move money, include an Idempotency-Key header with a client-generated unique string such as a UUID v4, so that a retried request returns the original response instead of creating a second charge."},
    {"id": "Q-03", "trap": None, "pages": ["rate-limits"],
     "question": "What is the rate limit in test mode?",
     "expected": "Test mode allows 25 requests per second. Over the limit, requests fail with HTTP 429, type rate_limit_error, and a Retry-After header giving the number of seconds to wait."},
    {"id": "Q-04", "trap": None, "pages": ["authentication"],
     "question": "After I rotate my secret key, how long does the old key keep working?",
     "expected": "The new key is active immediately and the old key keeps working for 24 hours so deployments can roll over. It can be revoked sooner from the dashboard if it was exposed."},
    {"id": "Q-05", "trap": None, "pages": ["refunds"],
     "question": "Can I refund part of a charge, and can I do it more than once?",
     "expected": "Yes. POST /v1/refunds with the charge id and an amount refunds part of it. A charge can be refunded several times until the refunded total reaches the original amount. A request beyond the remaining balance returns 400 with code amount_exceeds_refundable."},
    {"id": "Q-06", "trap": None, "pages": ["errors"],
     "question": "Should I retry a request that came back with a 5xx error?",
     "expected": "Yes. A request that returned 5xx may have partially completed, so retry it with exponential backoff and quote the request_id if you contact support. Do not retry 4xx errors other than 429."},
    {"id": "Q-07", "trap": None, "pages": ["quickstart"],
     "question": "Which test card numbers give me a successful payment and a decline?",
     "expected": "Use 4242 4242 4242 4242 for a successful payment and 4000 0000 0000 0002 for a decline."},
    {"id": "Q-08", "trap": None, "pages": ["webhooks-overview"],
     "question": "If my webhook endpoint is down, how long will Halcyon keep retrying deliveries?",
     "expected": "Deliveries are retried with exponential backoff, at 1 minute, 5 minutes, 30 minutes, 2 hours and then every 6 hours, for up to 72 hours. After that the event is marked failed and shown in the dashboard."},
    {"id": "Q-09", "trap": None, "pages": ["charges-create"],
     "question": "How do I authorize a card now and capture the payment later?",
     "expected": "Pass capture=false when creating the charge to authorize only, then capture within 7 days with POST /v1/charges/{id}/capture."},
    {"id": "Q-10", "trap": None, "pages": ["changelog-2026-03", "webhook-signatures-v2"],
     "question": "What changed for webhook signatures in API version 2026-03-01?",
     "expected": "Webhook signatures v2, HMAC-SHA256 with a timestamped Halcyon-Signature header, became the default. v1, HMAC-SHA1 with the X-Halcyon-Signature header, is deprecated and is sent only to accounts pinned to earlier versions. SDK 3.0 verifies v2 only and SDK 2.x verifies v1 only."},
]
QUESTION_BY_ID = {}
for question in DOC_QUESTIONS:
    QUESTION_BY_ID[question["id"]] = question

def question_fingerprint(questions):
    return hashlib.sha256(json.dumps(questions, sort_keys=True).encode()).hexdigest()[:12]

assert corpus_fingerprint(DOCS, TICKETS) == "e4485ba0f187", "DOCS or TICKETS drifted from post 0"
print(f"corpus fingerprint    {corpus_fingerprint(DOCS, TICKETS)}   the canonical pages and tickets, untouched")
print(f"question fingerprint  {question_fingerprint(DOC_QUESTIONS)}   {len(DOC_QUESTIONS)} questions, new in this post\n")
print(f"{'id':<6}{'trap':<21}{'pages that answer it':<42}question")
for question in DOC_QUESTIONS:
    trap = question["trap"] or ""
    print(f"{question['id']:<6}{trap:<21}{', '.join(question['pages']):<42}{question['question']}")


corpus fingerprint    e4485ba0f187   the canonical pages and tickets, untouched
question fingerprint  68e3908a66a7   10 questions, new in this post

id    trap                 pages that answer it                      question
Q-01  deprecated-page      webhook-signatures-v2                     How do I verify the signature on a webhook Halcyon sends me?
Q-02  missing-requirement  charges-create, api-conventions           How do I create a charge?
Q-03                       rate-limits                               What is the rate limit in test mode?
Q-04                       authentication                            After I rotate my secret key, how long does the old key keep working?
Q-05                       refunds                                   Can I refund part of a charge, and can I do it more than once?
Q-06                       errors                                    Should I retry a request that came back with a 5xx error?
Q-07                       quickstart       

In [6]:
# Carried forward from the retrieval post: the graph state, the inline retriever, the assistant, ask() and ask_all(), unchanged.
class DocsState(TypedDict):
    question: str
    retrieved: list[dict]        # chunk records: part, slug, status, updated, text, vector
    context: list[str] | None    # the chunk texts, which is all the generator sees
    answer: str

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def inline_retriever(top_k, highest_first=True):
    """A retriever over CHUNKS. highest_first=False reproduces a distance-for-similarity bug: the k chunks least like the question."""
    def retrieve_chunks(question_text):
        query_vector = embedder.embed_text(question_text)
        similarity = {}                                    # similarity[chunk part] -> cosine to the question
        for chunk in CHUNKS:
            similarity[chunk["part"]] = cosine(query_vector, chunk["vector"])
        ranked = sorted(CHUNKS, key=lambda chunk: similarity[chunk["part"]], reverse=highest_first)
        return ranked[:top_k]
    return retrieve_chunks

def build_assistant(retrieve_chunks):
    """Two nodes. The retriever passed in, then the setup cell's respond() as the generator."""
    def retrieve(state: DocsState) -> dict:
        retrieved = retrieve_chunks(state["question"])
        context = []
        for chunk in retrieved:
            context.append(chunk["text"])                  # status and updated stay on the record, not in the text
        return {"retrieved": retrieved, "context": context}

    builder = StateGraph(DocsState)
    builder.add_node("retrieve", retrieve)
    builder.add_node("generate", respond)
    builder.add_edge(START, "retrieve")
    builder.add_edge("retrieve", "generate")
    builder.add_edge("generate", END)
    return builder.compile()

def ask(assistant, question):
    """One question through one assistant. The result keeps the question record, the chunks retrieved and the answer."""
    state = assistant.invoke({"question": question["question"]})
    return {"id": question["id"], "question": question, "retrieved": state["retrieved"],
            "context": state["context"], "answer": state["answer"]}

def ask_all(assistant, questions):
    """Every question through one assistant, MAX_CONCURRENCY at a time, as post 0 ran the tickets. Keyed by question id."""
    results = {}
    with ThreadPoolExecutor(MAX_CONCURRENCY) as pool:
        futures = []
        for question in questions:
            futures.append(pool.submit(ask, assistant, question))
        for future in futures:
            result = future.result()
            results[result["id"]] = result
    return results

V1_SCHEME = re.compile(r"HMAC-SHA1\b|X-Halcyon-Signature", re.I)
V2_SCHEME = re.compile(r"HMAC-SHA256|300 seconds", re.I)

ASSISTANT = build_assistant(inline_retriever(TOP_K))
t0 = time.time()
RESULTS = {"healthy": ask_all(ASSISTANT, DOC_QUESTIONS)}   # RESULTS[config][question id] -> result
print(f"{len(RESULTS['healthy'])} questions answered in {time.time() - t0:.0f} s\n")

webhook = RESULTS["healthy"]["Q-01"]
print(webhook["question"]["question"])
print("-" * 78)
for rank, chunk in enumerate(webhook["retrieved"], 1):
    print(f"rank {rank}  {chunk['part']:<26}status {chunk['status']:<11}updated {chunk['updated']}")
print("-" * 78)
print(webhook["answer"])
print("-" * 78)
names_v1 = bool(V1_SCHEME.search(webhook["answer"]))
names_v2 = bool(V2_SCHEME.search(webhook["answer"]))
print(f"answer names the v1 scheme, HMAC-SHA1 or X-Halcyon-Signature: {names_v1}")
print(f"answer names the v2 scheme, HMAC-SHA256 or the 300 second rule: {names_v2}")
print(f"app model calls: {len(USAGE)}   embed calls: {len(EMBED_USAGE)}   judge calls: {judge_calls()}")


10 questions answered in 11 s

How do I verify the signature on a webhook Halcyon sends me?
------------------------------------------------------------------------------
rank 1  webhook-signatures-v1#1   status deprecated updated 2026-03-01
rank 2  webhook-signatures-v2#1   status current    updated 2026-03-01
rank 3  webhook-signatures-v1#2   status deprecated updated 2026-03-01
------------------------------------------------------------------------------
Based on the provided documentation, signature verification depends on the signature scheme used for your API version:

### 1. Webhook Signatures v2 (`Halcyon-Signature` header)
* **Header format:** `t=<unix timestamp>,v2=<hex digest>`
* **Signed payload:** Concatenate the timestamp, a period (`.`), and the exact raw request body.
* **Algorithm:** Compute an HMAC-SHA256 over the signed payload.

### 2. Legacy Scheme (`X-Halcyon-Signature` header)
*Used for accounts pinned to API versions before 2026-03-01 (deprecated on 2026-03-01)

The deprecated page ranked first again, the same three chunks in the same order as the retrieval post. This run's answer gives both schemes. It labels the old one legacy, for API versions before 2026-03-01. Hold onto that label. The rule we are about to write turns on it.

One more carried cell. The metric-library post's four money tickets, summarized again by the unchanged assistant. And its one-sentence G-Eval for the P1 rule. That post found the criterion put the kept summary above the three lost ones on every run. It also found the judge rewriting its steps each time. We start there.

In [7]:
# Carried forward from the metric-library post: its money-ticket summaries, the cases built from them,
# and its one-sentence GEval, assembled from two of its cells with its own import lines.
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

MONEY_TICKETS = [ticket for ticket in TICKETS if ticket["id"] in MONEY_TICKET_IDS]
GOLDEN_BY_ID = {golden.name: golden for golden in GOLDENS}

SUMMARIES = []      # four fresh runs of the unchanged assistant
for ticket in MONEY_TICKETS:
    SUMMARIES.append(triage(ticket))

CASES = {}
for result in SUMMARIES:
    golden = GOLDEN_BY_ID[result["id"]]
    CASES[result["id"]] = LLMTestCase(
        name=result["id"],
        input=golden.input,                       # the ticket
        actual_output=result["summary"],          # what the assistant wrote
        expected_output=golden.expected_output,   # the hand-written reference summary
        retrieval_context=[golden.input],         # what the summarizer was given: the ticket
        context=[golden.input],                   # the ground truth we vouch for: also the ticket
    )

KEPT = []     # tickets whose summary kept the money
LOST = []     # tickets whose summary lost it
for result in SUMMARIES:
    if MONEY.search(result["summary"]):
        KEPT.append(result["id"])
    else:
        LOST.append(result["id"])
assert KEPT and LOST, "the hunt needs at least one summary that kept the money and one that lost it"

print(f"{'ticket':<8}{'expected':<9}{'got':<5}{'money':<7}summary")
for result in SUMMARIES:
    expected = GOLDEN_BY_ID[result["id"]].additional_metadata["expected_severity"]
    money = "kept" if result["id"] in KEPT else "lost"
    print(f"{result['id']:<8}{expected:<9}{result['severity']:<5}{money:<7}{result['summary']}")
print(f"\npass mark: a metric hits if every kept summary {KEPT} scores above every lost summary {LOST}, on every run")

CRITERIA = ("The summary must keep any report in the ticket that funds were affected, such as duplicate "
            "charges, however briefly the ticket mentions it.")   # Halcyon's P1 rule, in one sentence
GEVAL_ARGS = dict(name="Money survives", criteria=CRITERIA,
                  evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT])
print(f"app model calls: {len(USAGE)}   judge calls: {judge_calls()}")


ticket  expected got  money  summary
T-1003  P1       P3   lost   The customer asks if pending webhook retries can be cancelled following a temporary 500 error outage on their endpoint.
T-1006  P1       P3   lost   After rolling back SDK 3.0 due to webhook verification errors, the merchant asks what changed and the required migration order.
T-1009  P1       P1   kept   Customer's import script times out around row 200, causing duplicate charges, and inquires about batch endpoints or recommended import concurrency.
T-1011  P1       P2   lost   The customer reports intermittent 502 HTML errors when making POST /v1/charges requests in the eu-west region since Tuesday.

pass mark: a metric hits if every kept summary ['T-1009'] scores above every lost summary ['T-1003', 'T-1006', 'T-1011'], on every run
app model calls: 18   judge calls: 0


Three summaries lost the money and one kept it, the same shape as before. Then everything new to this post, in one cell. Two imports, the number of runs, a pass mark of 0.5, and two patterns. `LABELLED` matches the words a deprecation label uses. `IDEMPOTENCY` matches the header name. `dag_path()` reads the per-node log DeepEval keeps on a DAG metric. It returns the path as a list of dicts, one per node visited, with the verdict and the judge's reason. The cell also finds the deprecated page by reading `DOCS`. So the rule below names a scheme the corpus really marks deprecated.

In [8]:
# New in this post. Everything above is carried forward unchanged from the earlier posts.
from deepeval.metrics import DAGMetric
from deepeval.metrics.dag import DeepAcyclicGraph, NonBinaryJudgementNode, BinaryJudgementNode

RUNS = (1, 2, 3, 4, 5)
PASS_MARK = 0.5                   # the threshold every metric in this post gets; a verdict is score >= PASS_MARK
LABELLED = re.compile(r"deprecat|legacy|before 2026-03-01", re.I)   # a deprecation label, as the house rule words it
IDEMPOTENCY = re.compile(r"idempotency", re.I)

def test_case(result):
    """A DeepEval test case from one result, as the retrieval post defined it: question, answer, reference answer and chunk texts."""
    return LLMTestCase(name=result["id"], input=result["question"]["question"], actual_output=result["answer"],
                       expected_output=result["question"]["expected"], retrieval_context=result["context"])

def dag_path(metric):
    """The nodes a measured DAG metric visited, in order. Each entry is a dict: node, verdict, reason. The last entry is the terminal verdict."""
    path = []
    for log in metric._verbose_steps:                    # DeepEval's own per-node log, kept on the metric
        node = re.search(r"^Label: (.*)$", log, re.M)
        verdict = re.search(r"^Verdict: (.*)$", log, re.M)
        reason = re.search(r"^Reason: (.*)$", log, re.M)
        ends_in = re.search(r"^Type: (.*)$", log, re.M)
        entry = {"node": "terminal", "verdict": verdict.group(1), "reason": ""}
        if node:
            entry["node"] = node.group(1)
        if reason:
            entry["reason"] = reason.group(1)
        if ends_in and ends_in.group(1) == "Deterministic":
            entry["reason"] = "a fixed score"
        elif ends_in:
            entry["reason"] = "hands over to " + ends_in.group(1)
        path.append(entry)
    return path

# The one deprecated page, read from DOCS rather than typed in. The house rule below names its scheme.
DEPRECATED_PAGES = []
for doc in DOCS:
    if doc["status"] == "deprecated":
        DEPRECATED_PAGES.append(doc)
assert len(DEPRECATED_PAGES) == 1, "the house rule below was written for one deprecated page"
V1_PAGE = DEPRECATED_PAGES[0]
assert "HMAC-SHA1" in V1_PAGE["body"] and "X-Halcyon-Signature" in V1_PAGE["body"]

print(f"deprecated pages in DOCS: {len(DEPRECATED_PAGES)}   {V1_PAGE['slug']}, {V1_PAGE['title']}, deprecated {V1_PAGE['updated']}")
print(f"runs per experiment: {len(RUNS)}   pass mark: {PASS_MARK}")
print(f"app model calls: {len(USAGE)}   judge calls: {judge_calls()}")


deprecated pages in DOCS: 1   webhook-signatures-v1, Webhook signatures (v1), deprecated 2026-03-01
runs per experiment: 5   pass mark: 0.5
app model calls: 18   judge calls: 0


## G-Eval writes its own marking scheme, every time

G-Eval comes from a 2023 paper by Liu and colleagues, and DeepEval's version keeps the paper's two steps. First the judge reads the criterion and writes a short list of evaluation steps. That is a marking scheme, three or four lines long. Then, in a second call, it follows those steps against the fields we named. It returns a score out of 10 with a reason, and DeepEval divides by 10. The steps live on the metric object. Measure the same object twice and the second measure reuses them. Build a fresh object and the judge writes fresh ones. Our `score()` helper builds a fresh metric per case. That is why every case on every run in the earlier posts got its own marking scheme.

The Summarization metric showed the same defect in the first metric post. There the judge regenerated its coverage questions on every run. Here the defect sits inside a metric people trust more, because the criterion is theirs.

The fix is one argument. `evaluation_steps` takes the list from us, and the judge skips the first call. So we build the P1 metric three times from the criterion and print the three step lists. Then we write our own three steps. We build it three times more, and print what comes back.

In [9]:
case = CASES["T-1003"]      # the lost summary the metric-library post walked through
print("ticket T-1003, summary:", case.actual_output)

print("\ncriteria only: three constructions of the same metric, each measured once")
before = judge_calls()
GENERATED = {}      # GENERATED[construction] -> the steps the judge wrote for that construction
for construction in (1, 2, 3):
    metric = GEval(model=judge, threshold=PASS_MARK, **GEVAL_ARGS)
    await metric.a_measure(case, _show_indicator=False)
    GENERATED[construction] = metric.evaluation_steps
    print(f"  construction {construction}: score {metric.score:.2f}, {len(metric.evaluation_steps)} steps")
    for number, step in enumerate(metric.evaluation_steps, 1):
        print(f"    {number}. {step}")
criteria_calls = judge_calls() - before
identical = GENERATED[1] == GENERATED[2] == GENERATED[3]
print(f"  the three step lists are identical: {identical}   judge calls: {criteria_calls}")

before = judge_calls()
await metric.a_measure(case, _show_indicator=False)   # the third object again: it keeps the steps it wrote
same_steps = metric.evaluation_steps == GENERATED[3]
print(f"\nthe third object measured a second time: score {metric.score:.2f}, steps unchanged {same_steps}, judge calls {judge_calls() - before}")

# The marking scheme, written by us once. The judge follows it and never rewrites it.
P1_STEPS = [
    "Read the ticket and note any report that funds were affected: a duplicate or double charge, a charge of the wrong amount, or a refund or payout that did not arrive, however briefly it is mentioned.",
    "Read the summary and check whether that report survives in it.",
    "Score 10 if the summary keeps every such report. Score 0 if the ticket has one and the summary leaves it out.",
]
P1_FIXED_ARGS = dict(name="Money survives", evaluation_steps=P1_STEPS, evaluation_params=GEVAL_ARGS["evaluation_params"])

print("\nour steps, passed in: three constructions, each measured once")
for number, step in enumerate(P1_STEPS, 1):
    print(f"    {number}. {step}")
before = judge_calls()
for construction in (1, 2, 3):
    metric = GEval(model=judge, threshold=PASS_MARK, **P1_FIXED_ARGS)
    await metric.a_measure(case, _show_indicator=False)
    steps_are_ours = metric.evaluation_steps == P1_STEPS
    print(f"  construction {construction}: score {metric.score:.2f}, steps are the ones we passed: {steps_are_ours}")
fixed_calls = judge_calls() - before
print(f"  judge calls: {fixed_calls}   ({criteria_calls} with criteria only)")


ticket T-1003, summary: The customer asks if pending webhook retries can be cancelled following a temporary 500 error outage on their endpoint.

criteria only: three constructions of the same metric, each measured once


  construction 1: score 0.20, 4 steps
    1. Scan the Input ticket for any mentions, however brief, indicating that funds were affected, such as duplicate charges or incorrect fees.
    2. Examine the Actual Output summary to identify whether it includes references to the affected funds reported in the Input.
    3. Compare the Actual Output against the Input to ensure no financial impact mentioned in the ticket was omitted from the summary.
    4. Determine if the Actual Output satisfies the criteria by successfully retaining all reports of affected funds found in the Input.


  construction 2: score 0.20, 3 steps
    1. Read the Input ticket to identify any mentions of funds being affected (such as duplicate charges), no matter how briefly reported.
    2. Check the Actual Output to verify if it includes the identified reports of affected funds from the Input.
    3. Ensure that no report of affected funds present in the Input is omitted in the Actual Output summary.


  construction 3: score 0.40, 4 steps
    1. Read the Input ticket to identify any mentions of affected funds, such as duplicate charges or financial discrepancies, regardless of how brief.
    2. Review the Actual Output (summary) to check if it includes the reports of affected funds identified in the Input.
    3. Compare the Input and Actual Output to verify that no reported financial impacts or affected funds from the ticket were omitted in the summary.
    4. Conclude whether the Actual Output meets the criteria based on the preservation of all fund-related reports from the Input.
  the three step lists are identical: False   judge calls: 6



the third object measured a second time: score 0.30, steps unchanged True, judge calls 1

our steps, passed in: three constructions, each measured once
    1. Read the ticket and note any report that funds were affected: a duplicate or double charge, a charge of the wrong amount, or a refund or payout that did not arrive, however briefly it is mentioned.
    2. Read the summary and check whether that report survives in it.
    3. Score 10 if the summary keeps every such report. Score 0 if the ticket has one and the summary leaves it out.


  construction 1: score 0.00, steps are the ones we passed: True


  construction 2: score 0.00, steps are the ones we passed: True


  construction 3: score 0.00, steps are the ones we passed: True
  judge calls: 3   (6 with criteria only)


Three constructions, three different marking schemes. The first wrote four steps, the second three, the third four again. No two lists match. The scores for the same lost summary were 0.20, 0.20 and 0.40. Then the third object measured again. One judge call, the steps unchanged, and the score moved anyway, to 0.30. The steps are only half the noise. The scoring call moves on its own.

Our three steps, passed in, came back three times as the three we wrote. One judge call each, three in total against six. The score was 0.00 on every construction. Our third step says score 0 when the money is gone, and the judge did what the step said. The criterion never said that. It said the summary must keep the money. It left the judge to decide what a miss is worth, and the judge decided four different things.

Halcyon's compliance lead asked for the same score tomorrow. Fixing the steps is the cheaper half of that. It is also where the rule stops being one sentence and becomes a marking scheme somebody owns. That is worth doing for every G-Eval in production. It is the first thing we carry forward.

## The rule no library ships, as one paragraph

Now the rule from the opening scene. We write it the way the docs team would, as one paragraph with three parts. The deprecation label, drawn from the page `DOCS` marks deprecated. The `Idempotency-Key` header, because the retrieval post found the charge answer omits it at top-k 3. And developer quality, because a rule with only prohibitions passes an empty answer. The metric reads the question, the answer and the chunks, and nothing we wrote by hand. It is referenceless, so it could run on the live queue.

We score this run's top-k 3 webhook answer with it first, and print the steps the judge wrote.

In [10]:
HOUSE_RULE = (
    "Halcyon deprecated its v1 webhook signature scheme, HMAC-SHA1 digests in an X-Halcyon-Signature header, on "
    "2026-03-01. The current scheme is v2, HMAC-SHA256 in a Halcyon-Signature header. The answer must not describe "
    "the v1 scheme unless it says that scheme is deprecated, legacy, or only for accounts pinned to API versions "
    "before 2026-03-01. If the question asks how to create a charge, the answer must tell the developer to send an "
    "Idempotency-Key header. Beyond those two rules, the answer should be a specific and correct developer answer "
    "to the question, drawn from the documentation supplied."
)
HOUSE_PARAMS = [SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT]
HOUSE_GEVAL_ARGS = dict(name="House rule", criteria=HOUSE_RULE, evaluation_params=HOUSE_PARAMS)

webhook_case = test_case(RESULTS["healthy"]["Q-01"])
names_v1 = bool(V1_SCHEME.search(webhook_case.actual_output))
labelled = bool(LABELLED.search(webhook_case.actual_output))
print(f"this run's top-k {TOP_K} webhook answer: names the v1 scheme {names_v1}, carries a deprecation label {labelled}")

before = judge_calls()
house = GEval(model=judge, threshold=PASS_MARK, **HOUSE_GEVAL_ARGS)
await house.a_measure(webhook_case, _show_indicator=False)
print(f"\nGEval, criteria only: score {house.score:.2f}   passes at {PASS_MARK}: {house.success}   judge calls {judge_calls() - before}")
print("steps the judge wrote from the rule:")
for number, step in enumerate(house.evaluation_steps, 1):
    print(f"  {number}. {step}")
print(f"reason: {house.reason}")
print(f"\nthe rule is {len(HOUSE_RULE.split())} words and asks for {len(house.evaluation_steps)} things at once: the deprecation label, the header, and quality")


this run's top-k 3 webhook answer: names the v1 scheme True, carries a deprecation label True



GEval, criteria only: score 1.00   passes at 0.5: True   judge calls 2
steps the judge wrote from the rule:
  1. Check if the Input asks how to create a charge, and if so, verify that the Actual Output explicitly instructs the developer to include an Idempotency-Key header.
  2. Verify that the Actual Output does not describe the v1 webhook signature scheme (HMAC-SHA1 in X-Halcyon-Signature) unless it clearly specifies that it is deprecated, legacy, or only for API versions pinned before 2026-03-01 in favor of v2.
  3. Ensure the Actual Output accurately and specifically answers the user's Input using correct developer details derived from the Retrieval Context.
reason: The output accurately answers how to verify Halcyon webhook signatures using the retrieved context. It correctly details the v2 scheme and properly identifies the v1 X-Halcyon-Signature scheme as legacy and deprecated for versions pinned before 2026-03-01, adhering to all evaluation steps.

the rule is 93 words and ask

The judge scored the answer 1.00. It wrote three steps from our 93-word paragraph, one per part. It put the header check first and the deprecation check second, the reverse of our order. Quality went into the third. Two judge calls. The reason says the answer labels the old scheme legacy, which it does. The pattern check above agreed. So far the paragraph works.

The trouble with one paragraph is that the judge decides how the three parts combine. Nothing in the rule says a broken deprecation label is worth zero. The judge could decide it is worth two points off. That is the question the rest of this post asks. The tree is the other answer to it.

### GEval

**Plain meaning:** Does the output meet a rule we wrote in plain language, judged in one reading?
**Under the hood:** From `criteria`, the judge writes three or four evaluation steps in one call. In a second call it follows them against the fields named in `evaluation_params`. It returns a score out of 10 with a reason, and DeepEval divides by 10. Pass `evaluation_steps` and the first call is skipped. The steps stay on the metric object. A fresh object gets fresh steps and a reused one keeps them. With an OpenAI judge that exposes token probabilities the score is smoothed across likely values. With our Gemini judge it is the number the judge wrote. `strict_mode` swaps in a prompt that asks for 1 or 0 and sets the threshold to 1. Two judge calls per test case, one with steps supplied.
**Needs:** whichever fields `evaluation_params` names, plus `criteria` or `evaluation_steps`. Referenceless unless `expected_output` is in the list. The P1 rule names `input` and `actual_output`. The house rule adds `retrieval_context`.
**Confused with:** `DAGMetric`. Both are custom. G-Eval is one judgement, and the judge produces the score. A DAG is several judgements in a fixed order, and we produce the score. The tell: if one part of the rule should end the evaluation on its own, that part is a gate. Gates are nodes.
**Use when:** the rule is one judgement of degree, like whether an answer is specific enough to act on.
**Do not use when:** the rule has a part that must never be traded against the rest. Below, the judge gives an answer that breaks the rule half marks. Half marks pass at the default threshold.

## The same rule as a tree

A DAG in DeepEval is a set of nodes we write and wire top down. A `NonBinaryJudgementNode` asks the judge to pick one label from a list we define. A `BinaryJudgementNode` asks for true or false. A `TaskNode` extracts something for later nodes and makes no decision. Each verdict on a judgement node is registered with `add_verdict`. A verdict either ends the path with a score from 0 to 10, or continues with `then`. The continuation is another node or a G-Eval. `DeepAcyclicGraph` checks the wiring when it is built. It rejects cycles. It rejects a binary node without exactly one true and one false verdict. It rejects a verdict with both a score and a continuation.

Our tree has two judgement nodes and one leaf. The first node classifies how the answer treats the v1 scheme: absent, labelled or unlabelled. Unlabelled ends the path at 0. The other two continue to the second node. That node asks whether the question is about creating a charge and, if so, whether the header is there. Key missing ends the path at 0. The other two continue to the leaf. The leaf is a G-Eval for developer quality with four steps we wrote. So the subjective question is asked last, and only of answers that kept both rules. An answer that breaks a rule never reaches the judge's opinion of its prose.

One wiring detail shapes the tree. When the graph is built, it counts each node's incoming edges. At run time a node runs only once that many verdicts have arrived. Two verdicts of the same node may share a child, and the graph counts them as one edge, because only one of them fires. Two verdicts of different nodes may not, in this version. The child waits for a second arrival that never comes, and the traversal ends without a score. The cell below builds the deprecation check both ways and asks each graph for its count. No judge call is involved.

In [ ]:
# New in this post. The wiring detail, read off the graph's own bookkeeping. No judge call: the graph counts each
# node's incoming edges when it is built, and these nodes never run, so their criteria are placeholders.
OUT = [SingleTurnParams.ACTUAL_OUTPUT]
uses_v1 = BinaryJudgementNode(criteria="uses the v1 scheme?", evaluation_params=OUT, label="uses v1")
is_labelled = BinaryJudgementNode(criteria="labelled deprecated or legacy?", evaluation_params=OUT, label="labelled")
key_check = BinaryJudgementNode(criteria="Idempotency-Key required?", evaluation_params=OUT, label="key")
uses_v1.add_verdict(False, then=key_check)
uses_v1.add_verdict(True, then=is_labelled)
is_labelled.add_verdict(False, score=0)
is_labelled.add_verdict(True, then=key_check)      # a second parent, owned by a different node
key_check.add_verdict(True, score=10)
key_check.add_verdict(False, score=0)
two_in_a_row = DeepAcyclicGraph(root_nodes=[uses_v1])

treatment = NonBinaryJudgementNode(criteria="absent, labelled or unlabelled?", evaluation_params=OUT, label="deprecated scheme")
key_check_2 = BinaryJudgementNode(criteria="Idempotency-Key required?", evaluation_params=OUT, label="key")
treatment.add_verdict("unlabelled", score=0)
treatment.add_verdict("absent", then=key_check_2)
treatment.add_verdict("labelled", then=key_check_2)   # a second parent, owned by the same node
key_check_2.add_verdict(True, score=10)
key_check_2.add_verdict(False, score=0)
three_way = DeepAcyclicGraph(root_nodes=[treatment])

print(f"two yes-or-no nodes in a row: the key node waits for {two_in_a_row.indegree[key_check]} arrivals, and one run delivers at most 1")
print(f"one three-way node:            the key node waits for {three_way.indegree[key_check_2]} arrival")


two yes-or-no nodes in a row: the key node waits for 2 arrivals, and one run delivers at most 1
one three-way node:            the key node waits for 1 arrival


Two yes-or-no nodes in a row leave the key node waiting for two arrivals. In any one run the first node sends it one, or the second does, never both. One three-way node leaves it waiting for one. That is why the deprecation check is a single node with three labels. The leaf G-Eval is exempt from all of this. It is not a node, so the graph never counts it, and any number of verdicts may hand over to it. It does need a model to construct, even though the tree swaps in its own at run time.

We score the same top-k 3 webhook answer, and print the path node by node.

In [11]:
# Node 1: the objective question first. One classification, three outcomes, and one of them ends the path at 0.
deprecated_node = NonBinaryJudgementNode(
    criteria=("Halcyon deprecated its v1 webhook signature scheme, HMAC-SHA1 digests in an X-Halcyon-Signature "
              "header. Classify how the answer treats that scheme. 'absent' if the answer never describes it. "
              "'labelled' if the answer describes it and says it is deprecated, legacy, or only for accounts pinned "
              "to API versions before 2026-03-01. 'unlabelled' if the answer describes it without any such label."),
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT], label="deprecated scheme")

# Node 2: the second objective question, only reached when the first did not end the path.
charge_node = NonBinaryJudgementNode(
    criteria=("Does the question ask how to create a charge? If it does not, reply 'not a charge question'. If it "
              "does, reply 'key required' when the answer tells the developer to send an Idempotency-Key header, "
              "and 'key missing' when it does not."),
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT], label="idempotency")

# Leaf: the subjective question last, as a GEval with steps we wrote. The DAG runs it only when both rules held.
QUALITY_STEPS = [
    "Read the question and identify what the developer needs to do or know.",
    "Check that the answer gives that, specifically, using the documentation supplied rather than inventing details.",
    "Where the documentation does not cover the question, check that the answer says so instead of guessing.",
    "Score 10 when the developer could act on the answer correctly, 5 when it is partly usable, 0 when it is unrelated or unusable.",
]
quality = GEval(name="quality", model=judge, evaluation_steps=QUALITY_STEPS,
                evaluation_params=HOUSE_PARAMS)   # needs a model to construct; the DAG hands it its own model at run time

deprecated_node.add_verdict("unlabelled", score=0)
deprecated_node.add_verdict("absent", then=charge_node)
deprecated_node.add_verdict("labelled", then=charge_node)
charge_node.add_verdict("key missing", score=0)
charge_node.add_verdict("not a charge question", then=quality)
charge_node.add_verdict("key required", then=quality)
HOUSE_DAG_ARGS = dict(name="House rule", dag=DeepAcyclicGraph(root_nodes=[deprecated_node]))

before = judge_calls()
tree = DAGMetric(model=judge, threshold=PASS_MARK, **HOUSE_DAG_ARGS)
await tree.a_measure(webhook_case, _show_indicator=False)
print(f"DAG on the same top-k {TOP_K} webhook answer: score {tree.score:.2f}   passes at {PASS_MARK}: {tree.success}   judge calls {judge_calls() - before}")
print("the path the judge took:")
for step in dag_path(tree):
    print(f"  {step['node']:<18}{step['verdict']:<24}{step['reason']}")
print(f"reason: {tree.reason}")


DAG on the same top-k 3 webhook answer: score 1.00   passes at 0.5: True   judge calls 3
the path the judge took:
  deprecated scheme labelled                The response describes the X-Halcyon-Signature HMAC-SHA1 scheme under the heading 'Legacy Scheme' and explicitly notes that it is used for accounts pinned to API versions before 2026-03-01 and was deprecated on 2026-03-01.
  idempotency       not a charge question   The question asks how to verify a webhook signature sent by Halcyon, not how to create a charge.
  terminal          not a charge question   hands over to quality [GEval] Metric
reason: The response accurately and comprehensively explains how to verify both Webhook Signatures v2 and legacy signatures using the details provided in the retrieval context. It correctly outlines the header formats, algorithms (HMAC-SHA256 and HMAC-SHA1), payload construction, and constant-time comparison requirement without inventing any information.


Three judge calls, one per node visited, and a score of 1.00 again. The path reads top down. The deprecation node said labelled and quoted the heading it found, "Legacy Scheme", with the pinned versions. The charge node said not a charge question. The terminal verdict handed over to the quality G-Eval, whose score and reason became the tree's. On this answer the paragraph and the tree agree, and both are right.

### DAGMetric

**Plain meaning:** A decision tree of small questions the judge answers in order. Every path ends at a score we chose or at another metric.
**Under the hood:** We build the nodes and register their verdicts. At measure time DeepEval walks the graph in dependency order, one judge call per node visited. A node's verdict picks the branch. A branch that ends at a fixed score costs one more call, to write a reason from the path. A branch that ends in a G-Eval runs that metric and adopts its score and reason. DeepEval divides the terminal score by 10. `strict_mode` sets the threshold to 1 and changes nothing in the graph, which the P1 section prints. Our tree visits two or three nodes per case, so three judge calls.
**Needs:** `input`, plus whatever fields the nodes name in `evaluation_params`. Referenceless unless a node names `expected_output`, which ours never do.
**Confused with:** `GEval`, above. The tell: a DAG's outcomes are fixed labels we typed, so its verdict can be printed node by node. A G-Eval's cannot. Also `ConversationalDAGMetric`, the multi-turn counterpart with the same node types over a whole conversation. A later post on multi-turn evaluation uses it.
**Use when:** the rule has stages, and a failed stage should score zero regardless of anything else.
**Do not use when:** the whole criterion is one judgement of degree. A tree of one node and two branches is G-Eval with more typing. Its one branch decision is still a judge call.

## Five runs, ten answers, three arms

A single agreement is a lead. So we score all ten of the retrieval post's answers five times with each of three arms. G-Eval from the paragraph. G-Eval from four steps we wrote for the same paragraph. And the tree. The second arm is the fix from the P1 section applied to the house rule. That way the tree is compared against the best G-Eval we know how to write, not the laziest.

Two axes. The spread is the highest score minus the lowest across the five runs, per question. The verdict is pass or fail at 0.5, and we count verdicts against a key. The key is ours, written by hand before the panel runs, by reading the ten answers against the rule. Pass unless a rule is broken. The two rules are checkable by pattern. So the cell stops if a regenerated answer disagrees with its label.

One label needs saying out loud. Q-09 asks how to authorize a card now and capture later. The answer is a `POST /v1/charges` with `capture=false`, a charge, and it never mentions the header. So the rule applies and the label is fail. In an earlier execution of this notebook we had labelled it pass, from the golden. A golden is the record we keep per question, with the reference answer we wrote by hand. Two of the three arms failed it. Reading the rule against the answer, the arms were right and our label was wrong. The golden's reference answer omits the header too. Nothing reference-based would ever have seen that.

In [12]:
# The key, by hand, before any panel call. Pass unless a house rule is broken. The two rules are checkable by
# pattern, so the cell stops if a regenerated answer disagrees with the label we wrote for it.
KEY = {   # KEY[question id] -> (expected verdict, why)
    "Q-01": (True,  "describes v1 and labels it legacy for accounts before 2026-03-01; gives v2 in full"),
    "Q-02": (False, "tells the developer how to create a charge and never mentions Idempotency-Key"),
    "Q-03": (True,  "25 requests per second, 429 and Retry-After, from the rate-limits page"),
    "Q-04": (True,  "24 hours, revoke sooner from the dashboard, from the authentication page"),
    "Q-05": (True,  "partial refunds, several times, amount_exceeds_refundable; the rule as written covers charges only"),
    "Q-06": (True,  "retry 5xx with backoff, quote request_id, do not retry 4xx except 429"),
    "Q-07": (True,  "the two test card numbers from the quickstart"),
    "Q-08": (True,  "the backoff schedule and the 72 hour limit from the webhooks page"),
    "Q-09": (False, "authorize-only is POST /v1/charges with capture=false, a charge; never mentions Idempotency-Key"),
    "Q-10": (True,  "v2 default, v1 deprecated, SDK 3.0 verifies v2 only; describes v1 as deprecated"),
}
CHARGE_QUESTIONS = ("Q-02", "Q-09")   # the questions whose answer is a POST /v1/charges, by our reading of the rule

for question in DOC_QUESTIONS:
    result = RESULTS["healthy"][question["id"]]
    print(f"{question['id']}  {question['question']}")
    print("-" * 78)
    print(result["answer"])
    print("-" * 78)
    names_v1 = bool(V1_SCHEME.search(result["answer"]))
    labelled = bool(LABELLED.search(result["answer"]))
    mentions_key = bool(IDEMPOTENCY.search(result["answer"]))
    expected, why = KEY[question["id"]]
    verdict = "pass" if expected else "FAIL"
    print(f"names v1 {names_v1}   deprecation label {labelled}   mentions Idempotency-Key {mentions_key}   key: {verdict}, {why}\n")
    if names_v1 and not labelled:
        assert expected is False, f"{question['id']}: this run's answer describes v1 without a label; relabel"
    if question["id"] in CHARGE_QUESTIONS:
        assert expected == mentions_key, f"{question['id']}: this run's answer mentions Idempotency-Key {mentions_key}; relabel"

expected_fails = []
for question_id in KEY:
    if KEY[question_id][0] is False:
        expected_fails.append(question_id)
print(f"expected to fail the house rule at top-k {TOP_K}: {expected_fails}   judge calls: {judge_calls()}")


Q-01  How do I verify the signature on a webhook Halcyon sends me?
------------------------------------------------------------------------------
Based on the provided documentation, signature verification depends on the signature scheme used for your API version:

### 1. Webhook Signatures v2 (`Halcyon-Signature` header)
* **Header format:** `t=<unix timestamp>,v2=<hex digest>`
* **Signed payload:** Concatenate the timestamp, a period (`.`), and the exact raw request body.
* **Algorithm:** Compute an HMAC-SHA256 over the signed payload.

### 2. Legacy Scheme (`X-Halcyon-Signature` header)
*Used for accounts pinned to API versions before 2026-03-01 (deprecated on 2026-03-01).*
* **Payload:** Compute a hex-encoded HMAC-SHA1 over the exact raw request body bytes (do **not** parse or re-serialize the JSON first).
* **Key:** Use your endpoint's signing secret (`whsec_`).
* **Comparison:** Compare your computed value against the `X-Halcyon-Signature` header using a constant-time comparison.

Ten answers, two labelled fail, both for the header. The pattern columns agree with every label. Q-01 and Q-10 describe the v1 scheme and both label it. One calls it legacy with the pinned versions, one calls it deprecated in a changelog. The other eight never mention it. Q-05 tells a developer how to post a refund without the header. The rule as written says charges, so it passes. We come back to that.

The next cell scores all of it. Our four steps for the house rule are at the top. Step 1 and step 2 each end with the words "the answer fails", and step 4 says score 0 for either. That is the best paragraph-shaped version of the rule we could write. It is the arm the tree has to beat. `PANEL[arm][run][question id]` holds every measured metric object, so the tables below can open any of them.

In [13]:
HOUSE_STEPS = [
    "Check whether the answer describes verification with HMAC-SHA1 or an X-Halcyon-Signature header. If it does without calling that scheme deprecated, legacy, or only for accounts pinned to API versions before 2026-03-01, the answer fails.",
    "If the question asks how to create a charge, check that the answer tells the developer to send an Idempotency-Key header. If it does not, the answer fails.",
    "Otherwise judge whether the answer is a specific and correct developer answer to the question, drawn from the documentation supplied.",
    "Score 0 for an answer that fails either rule. Otherwise score 0 to 10 for how well a developer could act on it.",
]
HOUSE_FIXED_ARGS = dict(name="House rule", evaluation_steps=HOUSE_STEPS, evaluation_params=HOUSE_PARAMS)

ARM_CLASS = {"GEval, criteria": GEval, "GEval, our steps": GEval, "DAG": DAGMetric}
ARM_ARGS = {"GEval, criteria": HOUSE_GEVAL_ARGS, "GEval, our steps": HOUSE_FIXED_ARGS, "DAG": HOUSE_DAG_ARGS}

HEALTHY_CASES = []
for question in DOC_QUESTIONS:
    HEALTHY_CASES.append(test_case(RESULTS["healthy"][question["id"]]))

# score() is the earlier posts' helper: MAX_CONCURRENCY cases in flight at once. This cell makes a few hundred
# judge calls, and one at a time would double its wall clock; the arms and runs are still sequential.
PANEL = {}            # PANEL[arm][run][question id] -> the measured metric object
CALLS_PER_CASE = {}   # judge calls one arm spends on one case, averaged over the panel
panel_start = judge_calls()
t0 = time.time()
for arm in ARM_CLASS:
    before = judge_calls()
    PANEL[arm] = {}
    for run in RUNS:
        PANEL[arm][run] = await score(ARM_CLASS[arm], HEALTHY_CASES, threshold=PASS_MARK, **ARM_ARGS[arm])
    CALLS_PER_CASE[arm] = (judge_calls() - before) / (len(RUNS) * len(HEALTHY_CASES))
panel_end = judge_calls()
print(f"{panel_end - panel_start} judge calls, {time.time() - t0:.0f} s\n")

def verdicts_of(arm, question_id):
    """The pass or fail verdict on one question for every run of one arm, as a string like PPFPP."""
    marks = ""
    for run in RUNS:
        if PANEL[arm][run][question_id].success:
            marks += "P"
        else:
            marks += "F"
    return marks

def right_verdicts(arm, question_id):
    """How many of the runs gave the verdict the key expects on this question."""
    expected = KEY[question_id][0]
    count = 0
    for run in RUNS:
        if PANEL[arm][run][question_id].success == expected:
            count += 1
    return count

run_headers = ""
for run in RUNS:
    run_headers += f"{'run ' + str(run):>7}"
for arm in ARM_CLASS:
    print(f"{arm}   ({CALLS_PER_CASE[arm]:.1f} judge calls per case)")
    print(f"  {'id':<6}{'key':<5}{run_headers}{'spread':>8}   verdicts  right")
    for question in DOC_QUESTIONS:
        question_id = question["id"]
        scores = []
        for run in RUNS:
            scores.append(PANEL[arm][run][question_id].score)
        spread = max(scores) - min(scores)
        expected = "pass" if KEY[question_id][0] else "FAIL"
        line = f"  {question_id:<6}{expected:<5}"
        for value in scores:
            line += f"{value:>7.2f}"
        line += f"{spread:>8.2f}   {verdicts_of(arm, question_id)}     {right_verdicts(arm, question_id)} of {len(RUNS)}"
        print(line)
    print()

SUMMARY = {}   # SUMMARY[arm] -> mean spread, questions with zero spread, questions where all runs agreed, verdicts right of 50, questions right on every run
for arm in ARM_CLASS:
    spreads = []
    zero_spread = 0
    agreed = 0
    right = 0
    right_every_run = 0
    for question in DOC_QUESTIONS:
        question_id = question["id"]
        scores = []
        for run in RUNS:
            scores.append(PANEL[arm][run][question_id].score)
        spreads.append(max(scores) - min(scores))
        if max(scores) == min(scores):
            zero_spread += 1
        marks = verdicts_of(arm, question_id)
        if marks.count(marks[0]) == len(marks):
            agreed += 1
        right += right_verdicts(arm, question_id)
        if right_verdicts(arm, question_id) == len(RUNS):
            right_every_run += 1
    SUMMARY[arm] = {"mean spread": statistics.mean(spreads), "zero spread": zero_spread, "agreed": agreed,
                    "right": right, "right every run": right_every_run}

total_verdicts = len(RUNS) * len(DOC_QUESTIONS)
print(f"{'arm':<18}{'mean spread':>12}{'zero spread':>13}{'all runs agree':>16}{'verdicts right':>16}{'right every run':>17}")
for arm in ARM_CLASS:
    row = SUMMARY[arm]
    print(f"{arm:<18}{row['mean spread']:>12.2f}{row['zero spread']:>10} of {len(DOC_QUESTIONS)}"
          f"{row['agreed']:>13} of {len(DOC_QUESTIONS)}{row['right']:>13} of {total_verdicts}{row['right every run']:>14} of {len(DOC_QUESTIONS)}")


300 judge calls, 322 s

GEval, criteria   (2.0 judge calls per case)
  id    key    run 1  run 2  run 3  run 4  run 5  spread   verdicts  right
  Q-01  pass    1.00   1.00   1.00   1.00   1.00    0.00   PPPPP     5 of 5
  Q-02  FAIL    0.50   0.60   0.50   0.50   0.60    0.10   PPPPP     0 of 5
  Q-03  pass    1.00   1.00   1.00   1.00   1.00    0.00   PPPPP     5 of 5
  Q-04  pass    1.00   1.00   1.00   1.00   1.00    0.00   PPPPP     5 of 5
  Q-05  pass    1.00   1.00   1.00   1.00   1.00    0.00   PPPPP     5 of 5
  Q-06  pass    1.00   1.00   1.00   1.00   1.00    0.00   PPPPP     5 of 5
  Q-07  pass    1.00   1.00   1.00   1.00   1.00    0.00   PPPPP     5 of 5
  Q-08  pass    1.00   1.00   1.00   1.00   1.00    0.00   PPPPP     5 of 5
  Q-09  FAIL    0.60   0.70   0.50   0.60   0.60    0.20   PPPPP     0 of 5
  Q-10  pass    1.00   1.00   1.00   1.00   1.00    0.00   PPPPP     5 of 5

GEval, our steps   (1.0 judge calls per case)
  id    key    run 1  run 2  run 3  run 4  run 5 

Three hundred judge calls, 322 seconds, and three tables. Read the verdict column against the key column.

The paragraph arm never failed anything. Q-02 scored 0.50 to 0.60 across the five runs, and Q-09 scored 0.50 to 0.70. Every one of those ten verdicts is a pass. The key says fail for both. The eight other questions scored 1.00 on every run. Mean spread 0.03, zero spread on eight questions, every run in agreement on all ten. Forty of fifty verdicts right, eight questions right on every run. This is the number to sit with. The paragraph arm was consistent. It agreed with itself on every question, and it was wrong on two of them every time. A score that does not move is only reassuring when it is also right.

Our-steps arm. Q-02 scored 0.00 five times. Q-09 scored 1.00, 0.00, 0.00, 0.00, 1.00, spread 1.00, verdicts PFFFP. Forty-eight of fifty right, nine questions right every run. The judge reading our steps had to decide whether an authorize-only charge is "how to create a charge". It decided differently on two runs out of five. The steps said what to score. They did not say what a charge question is. The judge filled that in, and not the same way twice.

The tree. Zero spread on all ten questions, fifty of fifty verdicts right, ten of ten right on every run. On Q-09 its charge node said key missing on every run. Same judge, same question, different framing. The node asks one thing, offers three labels, and the label picks the score. The paragraph asked the same judge to weigh a missing header against a correct capture flow. The judge took three to five points off. Our steps said zero, and the judge mostly agreed. The tree gave the judge nothing to weigh.

The judge calls per case are 2.0, 1.0 and 3.0. The tree costs three times the fixed-steps G-Eval and gets two more verdicts right out of fifty. Whether that is worth it depends on what the two are. Here they were the two runs where an answer that leads to a double charge would have passed.

## The answer that scored 1.00, finally failing

Here is the answer from the retrieval post's opening scene, verbatim. It came from the same retriever at top-k 2. Rank 1 was the deprecated SHA1 chunk and rank 2 the first chunk of the current page. The deprecation notice was nowhere. The answer gives the SHA1 recipe in full. For the current scheme it gives the header format. Then it says the documentation does not include the hashing algorithm. Faithfulness scored it 1.00 there, because it contradicted nothing it was given. Answer Relevancy scored it 1.00, because it answered the question. The pattern check above says the same thing three ways. Names v1, does not name v2, carries no deprecation label.

We score it five times with each arm, and print the tree's path on the first run in full.

In [14]:
# New in this post. Only the K2_WEBHOOK_ANSWER string is carried forward: the retrieval post's top-k 2 webhook
# answer, verbatim from that notebook's output, with the two chunks it was given. It scored 1.00 on Faithfulness there.
K2_WEBHOOK_ANSWER = """Based on the supplied documentation, signature verification depends on which header your webhook request contains:

### For `X-Halcyon-Signature`:
1. Get the raw request body exact bytes **before** any JSON parsing.
2. Compute a hex-encoded HMAC-SHA1 of the raw request body using your endpoint's signing secret (which starts with `whsec_`).
3. Compare the computed value with the `X-Halcyon-Signature` header using a constant-time comparison.
4. Reject the event if the values differ.

### For `Halcyon-Signature`:
* The header format is `t=<unix timestamp>,v2=<hex digest>`.
* The payload to verify/sign is constructed as: `<unix timestamp>.<raw request body>`.

*(Note: The provided documentation does not include further details on the specific hashing algorithm or comparison steps for the `v2` header).*"""
K2_PARTS = ["webhook-signatures-v1#1", "webhook-signatures-v2#1"]

k2_context = []
for chunk in CHUNKS:
    if chunk["part"] in K2_PARTS:
        k2_context.append(chunk["text"])
assert len(k2_context) == 2
k2_case = LLMTestCase(name="Q-01 at top-k 2", input=QUESTION_BY_ID["Q-01"]["question"], actual_output=K2_WEBHOOK_ANSWER,
                      expected_output=QUESTION_BY_ID["Q-01"]["expected"], retrieval_context=k2_context)
print(f"top-k 2 webhook answer: names the v1 scheme {bool(V1_SCHEME.search(K2_WEBHOOK_ANSWER))}, "
      f"names the v2 scheme {bool(V2_SCHEME.search(K2_WEBHOOK_ANSWER))}, carries a deprecation label {bool(LABELLED.search(K2_WEBHOOK_ANSWER))}")

PAYOFF = {}      # PAYOFF[arm][run] -> the measured metric object, the top-k 2 case scored five times per arm
before = judge_calls()
t0 = time.time()
for arm in ARM_CLASS:
    PAYOFF[arm] = {}
    for run in RUNS:
        metric = ARM_CLASS[arm](model=judge, threshold=PASS_MARK, **ARM_ARGS[arm])
        await metric.a_measure(k2_case, _show_indicator=False)
        PAYOFF[arm][run] = metric
print(f"{judge_calls() - before} judge calls, {time.time() - t0:.0f} s\n")

print("the DAG's path on run 1, node by node")
for step in dag_path(PAYOFF["DAG"][1]):
    print(f"  {step['node']:<18}{step['verdict']:<24}{step['reason']}")
print(f"  reason: {PAYOFF['DAG'][1].reason}")

print(f"\n{'arm':<18}{run_headers}   verdicts   top-k 3 answer, from the panel")
for arm in ARM_CLASS:
    line = f"{arm:<18}"
    marks = ""
    for run in RUNS:
        line += f"{PAYOFF[arm][run].score:>7.2f}"
        if PAYOFF[arm][run].success:
            marks += "P"
        else:
            marks += "F"
    line += f"   {marks}        {verdicts_of(arm, 'Q-01')}"
    print(line)

print("\nGEval, criteria only: what the judge wrote on each run")
for run in RUNS:
    metric = PAYOFF["GEval, criteria"][run]
    print(f"  run {run}  {metric.score:.2f}  {metric.reason}")


top-k 2 webhook answer: names the v1 scheme True, names the v2 scheme False, carries a deprecation label False


25 judge calls, 91 s

the DAG's path on run 1, node by node
  deprecated scheme unlabelled              The answer describes the X-Halcyon-Signature header and its HMAC-SHA1 verification process, but it does not mention that it is deprecated, legacy, or restricted to specific API versions.
  terminal          unlabelled              a fixed score
  reason: The score is 0.0 because the NonBinaryJudgementNode evaluated the answer under the 'deprecated scheme' criteria and assigned a verdict of 'unlabelled' (since the answer described the X-Halcyon-Signature header without specifying that it is deprecated or legacy), which directly led to the final VerdictNode of 'unlabelled'.

arm                 run 1  run 2  run 3  run 4  run 5   verdicts   top-k 3 answer, from the panel
GEval, criteria      0.60   0.40   0.60   0.50   0.60   PFPPP        PPPPP
GEval, our steps     0.00   0.00   0.00   0.00   0.00   FFFFF        PPPPP
DAG                  0.00   0.00   0.00   0.00   0.00   FFFFF       

The tree's path is one node long. The deprecation node said unlabelled, and its reason says why. The answer describes the `X-Halcyon-Signature` header and its HMAC-SHA1 verification without saying the scheme is deprecated or legacy. The path ended at the fixed score. The reason call wrote the same thing back in a sentence. Zero on every run. The answer that scored 1.00 on both generator metrics scored 0.00 five times. The tree can say which node did it.

The paragraph arm scored it 0.60, 0.40, 0.60, 0.50 and 0.60. Four passes and one fail. Now read the five reasons. Every one of them names the violation. Run 1: it "fails to meet the evaluation requirement of framing the v1 signature scheme as deprecated or legacy". Run 4: "it fails step 2". The judge saw the broken rule on every run. It wrote it down on every run. It passed the answer on four of them. In the four passing reasons the violation comes after the word "however". In the one failing reason, the praise does. The answer accurately reflects the retrieval context, however it breaks the rule. On run 1 that was six points for accuracy and four off for the rule. That is what a bundled criterion buys. The judge treated Halcyon's hard rule as a deduction, and the deduction was smaller than the credit.

There is a threshold argument to face here. The paragraph arm scored this answer between 0.40 and 0.60. In the panel it scored the two broken answers no higher than 0.70, and every kept one at 1.00. A threshold of 0.8 would have separated them on this set. It would also have been a number chosen after reading these scores. It would be calibrated to how many points this judge takes off for this rule on these answers. A longer answer that did more things right would earn back the deduction. The tree does not have a threshold for the hard part. Zero is zero.

Our-steps arm scored it 0.00 on all five runs. On this case the marking scheme did the tree's work. Step 1 says the answer fails, and the judge followed it. On this post's cases the two arms differed on one question, Q-09. There the steps left a definition to the judge, and the tree did not.

## The same rule at two values of k

The retrieval post found that the charge trap is a retrieval failure. At top-k 3 the chunk with the header rule is not retrieved, and the answer omits it. At top-k 4 the chunk is retrieved, and the answer includes it. That gives the criterion a test it has to pass. A metric that encodes the rule must fail the top-k 3 answer and pass the top-k 4 answer, on every run. If it can only do one of those, it is measuring retrieval, not the rule.

The top-k 3 answer is this run's Q-02, printed in the key above. The top-k 4 answer comes from the retrieval post's wider retriever, one chunk more. We print it in full. Then five runs per arm on each.

In [15]:
WIDER = build_assistant(inline_retriever(4))      # the retrieval post's wider retriever, one chunk more
charge_k3 = RESULTS["healthy"]["Q-02"]
charge_k4 = ask(WIDER, QUESTION_BY_ID["Q-02"])
k3_mentions = bool(IDEMPOTENCY.search(charge_k3["answer"]))
k4_mentions = bool(IDEMPOTENCY.search(charge_k4["answer"]))
k3_parts = []
for chunk in charge_k3["retrieved"]:
    k3_parts.append(chunk["part"])
k4_parts = []
for chunk in charge_k4["retrieved"]:
    k4_parts.append(chunk["part"])
print(f"top-k 3 retrieved {', '.join(k3_parts)}   answer mentions Idempotency-Key {k3_mentions}")
print(f"top-k 4 retrieved {', '.join(k4_parts)}   answer mentions Idempotency-Key {k4_mentions}")
assert not k3_mentions and k4_mentions, "the charge trap did not fire the way the retrieval post found"

print("\ntop-k 4 answer")
print("-" * 78)
print(charge_k4["answer"])
print("-" * 78)

K_CASES = {"top-k 3": test_case(charge_k3), "top-k 4": test_case(charge_k4)}
K_TEST = {}      # K_TEST[arm][k label][run] -> the measured metric object
before = judge_calls()
t0 = time.time()
for arm in ARM_CLASS:
    K_TEST[arm] = {}
    for k_label, k_case in K_CASES.items():
        K_TEST[arm][k_label] = {}
        for run in RUNS:
            metric = ARM_CLASS[arm](model=judge, threshold=PASS_MARK, **ARM_ARGS[arm])
            await metric.a_measure(k_case, _show_indicator=False)
            K_TEST[arm][k_label][run] = metric
print(f"\n{judge_calls() - before} judge calls, {time.time() - t0:.0f} s\n")

print(f"{'arm':<18}{'k':<9}{run_headers}   verdicts")
for arm in ARM_CLASS:
    for k_label in K_CASES:
        line = f"{arm:<18}{k_label:<9}"
        marks = ""
        for run in RUNS:
            line += f"{K_TEST[arm][k_label][run].score:>7.2f}"
            if K_TEST[arm][k_label][run].success:
                marks += "P"
            else:
                marks += "F"
        print(line + f"   {marks}")

print("\nfails top-k 3 and passes top-k 4, on every run")
for arm in ARM_CLASS:
    holds = True
    for run in RUNS:
        if K_TEST[arm]["top-k 3"][run].success or not K_TEST[arm]["top-k 4"][run].success:
            holds = False
    print(f"  {arm:<18}{holds}")

print("\nthe DAG's idempotency node on the top-k 3 answer, every run")
for run in RUNS:
    for step in dag_path(K_TEST["DAG"]["top-k 3"][run]):
        if step["node"] == "idempotency":
            print(f"  run {run}  {step['verdict']:<14}{step['reason']}")


top-k 3 retrieved quickstart#2, charges-create#1, charges-create#2   answer mentions Idempotency-Key False
top-k 4 retrieved quickstart#2, charges-create#1, charges-create#2, api-conventions#2   answer mentions Idempotency-Key True

top-k 4 answer
------------------------------------------------------------------------------
Based on the provided documentation, you create a charge by making a `POST /v1/charges` request.

**Required Headers:**
* `Idempotency-Key`: A client-generated unique string (a UUID v4 is recommended).
* `Halcyon-Version`: The API version you tested against.

**Required Body Parameters:**
* `amount`: An integer in minor units (e.g., `1999` for 19.99).
* `currency`: A lowercase ISO 4217 code (e.g., `usd` or `eur`).
* `source`: A token from Halcyon.js or a saved payment method ID starting with `pm_`.

**Optional Body Parameters:**
* `description`: A description for the charge.
* `metadata`: Up to 20 key-value pairs.
* `capture`: A boolean that defaults to `true`. Pas


60 judge calls, 205 s

arm               k          run 1  run 2  run 3  run 4  run 5   verdicts
GEval, criteria   top-k 3     0.50   0.50   0.50   0.50   0.50   PPPPP
GEval, criteria   top-k 4     1.00   1.00   1.00   1.00   1.00   PPPPP
GEval, our steps  top-k 3     0.00   0.00   0.00   0.00   0.00   FFFFF
GEval, our steps  top-k 4     1.00   1.00   1.00   1.00   1.00   PPPPP
DAG               top-k 3     0.00   0.00   0.00   0.00   0.00   FFFFF
DAG               top-k 4     1.00   1.00   1.00   1.00   1.00   PPPPP

fails top-k 3 and passes top-k 4, on every run
  GEval, criteria   False
  GEval, our steps  True
  DAG               True

the DAG's idempotency node on the top-k 3 answer, every run
  run 1  key missing   The input question asks how to create a charge, but the provided response does not instruct the developer to send an Idempotency-Key header.
  run 2  key missing   The question asks how to create a charge, but the provided output does not instruct the developer to inc

The trap fired the way the retrieval post found. Top-k 3 retrieved the quickstart and the two charge chunks, and the answer never mentions the header. Top-k 4 added `api-conventions#2`. The answer grew a required header, `Idempotency-Key`, with the UUID recommendation. The two answers differ by that one line and some heading wording.

The tree failed the first on every run and passed the second on every run. Its charge node said key missing five times. The five reasons are the same sentence with the words moved. Our-steps G-Eval did the same, 0.00 five times and 1.00 five times. The paragraph arm gave the top-k 3 answer 0.50 on every run, exactly the pass line, and passed it five times. At the default threshold, the paragraph arm cannot tell the answer that causes double charges from the one that prevents them.

Sixty judge calls, 205 seconds. Two of the three arms encode the rule. One of them encodes retrieval quality with a deduction for the rule. That is the thing we were trying to stop measuring.

## The P1 rule, briefly, as a tree

The second criterion is the one the series already had. Halcyon's SLA, its support response-time policy, says any ticket reporting affected funds is P1. The metric-library post found a one-sentence G-Eval that put the kept summary above the three lost ones on every run. We build it three ways over the four money tickets. G-Eval with our three steps from the first section. The same steps in `strict_mode`, which asks the judge for 1 or 0 and sets the threshold to 1. And a tree of two nodes. The first reads only the ticket and asks whether it reports affected funds. If not, the path ends at 10, so a ticket with no money in it cannot fail. If so, the second node reads ticket and summary and asks whether the report survived. True ends at 10, false at 0.

Five runs each, and the pass mark from the metric-library post. Kept above lost, on every run.

In [16]:
# The P1 rule as a tree. Node 1 reads the ticket alone; node 2 reads ticket and summary, and only runs when node 1 found money.
ticket_node = NonBinaryJudgementNode(
    criteria=("Does the ticket report that funds were affected: a duplicate or double charge, a charge of the wrong "
              "amount, or a refund or payout that did not arrive, however briefly? Reply 'reports funds affected' "
              "or 'no funds report'."),
    evaluation_params=[SingleTurnParams.INPUT], label="ticket")
summary_node = BinaryJudgementNode(
    criteria="Does the summary keep the ticket's report that funds were affected?",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT], label="summary")
ticket_node.add_verdict("no funds report", score=10)
ticket_node.add_verdict("reports funds affected", then=summary_node)
summary_node.add_verdict(True, score=10)
summary_node.add_verdict(False, score=0)
P1_DAG_ARGS = dict(name="Money survives", dag=DeepAcyclicGraph(root_nodes=[ticket_node]))
P1_STRICT_ARGS = dict(name="Money survives", evaluation_steps=P1_STEPS, evaluation_params=GEVAL_ARGS["evaluation_params"],
                      strict_mode=True)   # the judge is asked for 1 or 0, and the threshold becomes 1

P1_CLASS = {"GEval, our steps": GEval, "GEval, strict": GEval, "DAG": DAGMetric}
P1_ARGS = {"GEval, our steps": P1_FIXED_ARGS, "GEval, strict": P1_STRICT_ARGS, "DAG": P1_DAG_ARGS}
MONEY_CASES = []
for ticket_id in MONEY_TICKET_IDS:
    MONEY_CASES.append(CASES[ticket_id])

P1_PANEL = {}    # P1_PANEL[arm][run][ticket id] -> the measured metric object
P1_CALLS = {}
before = judge_calls()
t0 = time.time()
for arm in P1_CLASS:
    arm_start = judge_calls()
    P1_PANEL[arm] = {}
    for run in RUNS:
        P1_PANEL[arm][run] = {}
        for case in MONEY_CASES:
            metric = P1_CLASS[arm](model=judge, threshold=PASS_MARK, **P1_ARGS[arm])   # strict_mode overrides the threshold with 1
            await metric.a_measure(case, _show_indicator=False)
            P1_PANEL[arm][run][case.name] = metric
    P1_CALLS[arm] = (judge_calls() - arm_start) / (len(RUNS) * len(MONEY_CASES))
print(f"{judge_calls() - before} judge calls, {time.time() - t0:.0f} s\n")

def hit(scores):
    """True when the lowest score among kept summaries is above the highest score among lost ones."""
    lowest_kept = min(scores[ticket_id].score for ticket_id in KEPT)
    highest_lost = max(scores[ticket_id].score for ticket_id in LOST)
    return lowest_kept > highest_lost

for arm in P1_CLASS:
    print(f"{arm}   ({P1_CALLS[arm]:.1f} judge calls per case, threshold {P1_PANEL[arm][1][MONEY_TICKET_IDS[0]].threshold})")
    print(f"  {'ticket':<8}{'money':<6}{run_headers}{'spread':>8}")
    for ticket_id in MONEY_TICKET_IDS:
        scores = []
        for run in RUNS:
            scores.append(P1_PANEL[arm][run][ticket_id].score)
        money = "kept" if ticket_id in KEPT else "lost"
        line = f"  {ticket_id:<8}{money:<6}"
        for value in scores:
            line += f"{value:>7.2f}"
        print(line + f"{max(scores) - min(scores):>8.2f}")
    per_run = []
    for run in RUNS:
        if hit(P1_PANEL[arm][run]):
            per_run.append("hit")
        else:
            per_run.append("miss")
    print(f"  kept above lost, per run: {' '.join(per_run)}\n")

print("the DAG's path on T-1003 and T-1009, run 1")
for ticket_id in ("T-1003", "T-1009"):
    print(f"  {ticket_id}")
    for step in dag_path(P1_PANEL["DAG"][1][ticket_id]):
        print(f"    {step['node']:<10}{step['verdict']:<24}{step['reason']}")

strict_tree = DAGMetric(model=judge, strict_mode=True, **P1_DAG_ARGS)
print(f"\nstrict_mode on a DAGMetric sets threshold {strict_tree.threshold} and changes nothing else in the graph")


100 judge calls, 269 s

GEval, our steps   (1.0 judge calls per case, threshold 0.5)
  ticket  money   run 1  run 2  run 3  run 4  run 5  spread
  T-1003  lost     0.00   0.00   0.00   0.00   0.00    0.00
  T-1006  lost     0.00   0.00   0.00   0.00   0.00    0.00
  T-1009  kept     1.00   1.00   1.00   1.00   1.00    0.00
  T-1011  lost     0.00   0.00   0.00   0.00   0.00    0.00
  kept above lost, per run: hit hit hit hit hit

GEval, strict   (1.0 judge calls per case, threshold 1)
  ticket  money   run 1  run 2  run 3  run 4  run 5  spread
  T-1003  lost     0.00   0.00   0.00   0.00   0.00    0.00
  T-1006  lost     0.00   0.00   0.00   0.00   0.00    0.00
  T-1009  kept     1.00   1.00   1.00   1.00   1.00    0.00
  T-1011  lost     0.00   0.00   0.00   0.00   0.00    0.00
  kept above lost, per run: hit hit hit hit hit

DAG   (3.0 judge calls per case, threshold 0.5)
  ticket  money   run 1  run 2  run 3  run 4  run 5  spread
  T-1003  lost     0.00   0.00   0.00   0.00   0.00  

All three arms hit on all five runs. Every score was 0.00 or 1.00 with zero spread. The tree separated the four summaries no more reliably than the fixed-steps G-Eval, because there was nothing left to separate. The plan for this post expected the tree to win here. It did not, and that is the result. With three steps whose last line says score 10 or 0, the judge already scores in whole numbers. Strict mode changes the prompt and the threshold, and changed nothing in the scores. The tree's path on T-1003 says what the G-Eval's reason said in the metric-library post, in two lines instead of one. The ticket reports funds affected, and the summary leaves it out.

One hundred judge calls, 269 seconds. The last line prints what `strict_mode` does to a tree. It sets the threshold to 1 and leaves the graph alone. On a G-Eval it also swaps the scoring prompt. On a DAG the scores were already ours.

## Accuracy and variance side by side, and when the extra work is worth it

Every case this post labelled, in one table. The ten questions at top-k 3, five runs each. The top-k 2 webhook answer, five runs. The two charge answers, five runs each. Sixty-five verdicts per arm.

In [17]:
ACCURACY = {}   # ACCURACY[arm] -> verdicts right over every case this post scored against a label
for arm in ARM_CLASS:
    right = SUMMARY[arm]["right"]
    total = total_verdicts
    for run in RUNS:
        total += 3                                        # the top-k 2 webhook answer and the two charge answers
        if not PAYOFF[arm][run].success:
            right += 1                                    # labelled fail: v1 in full and unlabelled
        if not K_TEST[arm]["top-k 3"][run].success:
            right += 1                                    # labelled fail: no Idempotency-Key
        if K_TEST[arm]["top-k 4"][run].success:
            right += 1                                    # labelled pass: the header is there
    ACCURACY[arm] = {"right": right, "total": total}

print(f"{'arm':<18}{'calls/case':>11}{'mean spread':>13}{'all runs agree':>16}{'verdicts right':>16}")
for arm in ARM_CLASS:
    row = SUMMARY[arm]
    accuracy = ACCURACY[arm]
    print(f"{arm:<18}{CALLS_PER_CASE[arm]:>11.1f}{row['mean spread']:>13.2f}{row['agreed']:>13} of {len(DOC_QUESTIONS)}"
          f"{accuracy['right']:>13} of {accuracy['total']}")

print("\nwhere each arm went wrong, by case, counted over the runs")
for arm in ARM_CLASS:
    wrong = {}
    for question in DOC_QUESTIONS:
        misses = len(RUNS) - right_verdicts(arm, question["id"])
        if misses:
            wrong[question["id"]] = misses
    for run in RUNS:
        if PAYOFF[arm][run].success:
            wrong["Q-01 at top-k 2"] = wrong.get("Q-01 at top-k 2", 0) + 1
        if K_TEST[arm]["top-k 3"][run].success:
            wrong["Q-02 at top-k 3"] = wrong.get("Q-02 at top-k 3", 0) + 1
        if not K_TEST[arm]["top-k 4"][run].success:
            wrong["Q-02 at top-k 4"] = wrong.get("Q-02 at top-k 4", 0) + 1
    print(f"  {arm:<18}{wrong}")


arm                calls/case  mean spread  all runs agree  verdicts right
GEval, criteria           2.0         0.03           10 of 10           46 of 65
GEval, our steps          1.0         0.10            9 of 10           63 of 65
DAG                       3.0         0.00           10 of 10           65 of 65

where each arm went wrong, by case, counted over the runs
  GEval, criteria   {'Q-02': 5, 'Q-09': 5, 'Q-01 at top-k 2': 4, 'Q-02 at top-k 3': 5}
  GEval, our steps  {'Q-09': 2}
  DAG               {}


Forty-six of sixty-five for the paragraph. Sixty-three for our steps. Sixty-five for the tree. The paragraph's nineteen misses are all the same miss. Four cases where a rule was broken, and it passed them on nineteen of twenty runs. Our-steps missed twice, both on Q-09. The tree missed nothing. Mean spread across the panel: 0.03, 0.10 and 0.00. The paragraph's spread is smaller than our-steps', and its accuracy is far worse. Spread measures whether the judge agrees with itself. The key measures whether it agrees with Halcyon. The first without the second is the auditor's question answered with the wrong number.

So when is the tree worth three calls where one would do? When the rule has a part that must score zero, and the judge would otherwise weigh it. On the house rule that part existed, and the tree earned its cost. On the P1 rule the fixed steps already scored in whole numbers. There the tree earned a path a support lead can read, and a rule change that is one node's criteria, and nothing in the numbers. Between those two sits a question every team answers per rule. Would a wrong answer to any one part of this criterion be a defect on its own? If yes, that part is a node. If the whole rule is one judgement of degree, write the steps, pass them in, and keep the two calls.

Two limits, stated plainly. The tree's zero spread came from a judge that read three nodes the same way ten times. Nothing in the judge is deterministic. DeepEval's documentation says so. Task and judgement nodes still use a model, and their outputs can vary. What the tree fixes is the map from answers to score. A harder gate, or a longer answer, could move a node's verdict, and then the tree would move too. And the rule as written covers charges only. Refunds and payouts are money-moving requests under the same API conventions page. Q-05's answer tells a developer how to post a refund without the header. Extending the rule is one node's criteria in the tree. In the paragraph it is another clause for the judge to weigh.

## What teams get wrong

**Writing one criterion that bundles several judgements.** Our 93-word paragraph asked for three things. The judge wrote three steps, then scored them as one number. A broken rule became a deduction. The answer that leads to double charges scored 0.50 on every run and passed. The fix is a shape, not better wording. Each thing that must be true on its own is a node. The one judgement of degree goes last.

**Believing a sampling setting makes a judge reproducible.** DeepEval sent temperature 0.0 on every call in this post, as its registry says it should for this model. Google's guide for the 3.x family says the parameter is deprecated and stripped. With the same steps and the same object, the P1 score moved from 0.40 to 0.30 between two calls. The judge moving is a property of the system. The only settings that reduced it here were structural. Steps passed in, and gates with fixed outcomes.

**Reaching for a tree when the criterion is one judgement.** The P1 tree cost three calls per case. It scored exactly what one call with three good steps scored. A tree of one gate is a G-Eval with the score written down, and its gate is still a judge call. Write the steps first. Add nodes only where a part of the rule needs to end the evaluation by itself.

**Trusting the key because it came from the goldens.** Our first label for Q-09 was pass, read off the reference answer. The reference answer was wrong by Halcyon's own rule. A hand-labelled key has to be labelled against the rule. Reading the answers is the only way to do it.

In [18]:
# Our ledger: every judge call the wrapper saw, thinking tokens included.
judge_in = 0
judge_visible = 0
judge_thinking = 0
for usage, temperature in JUDGE_USAGE:
    judge_in += usage.prompt_token_count
    judge_visible += usage.candidates_token_count
    judge_thinking += usage.thoughts_token_count or 0
judge_out = judge_visible + judge_thinking
judge_cost = judge_in * PRICE_IN + judge_out * PRICE_OUT
temperatures = []
for usage, temperature in JUDGE_USAGE:
    if temperature not in temperatures:
        temperatures.append(temperature)

# DeepEval's own counter for the panel, against our ledger for the same calls.
deepeval_in = 0
deepeval_cost = 0
for arm in PANEL:
    for run in PANEL[arm]:
        for metric in PANEL[arm][run].values():
            deepeval_in += metric.input_tokens
            deepeval_cost += metric.evaluation_cost
panel_in = 0
panel_out = 0
for usage, temperature in JUDGE_USAGE[panel_start:panel_end]:
    panel_in += usage.prompt_token_count
    panel_out += usage.candidates_token_count + (usage.thoughts_token_count or 0)
panel_cost = panel_in * PRICE_IN + panel_out * PRICE_OUT

# The assistant's own calls, recorded by the setup cell, and the embedding calls, recorded by embed().
app_in = 0
app_out = 0
for usage in USAGE:
    app_in += usage.get("input_tokens", 0)
    app_out += usage.get("output_tokens", 0)
app_cost = app_in * PRICE_IN + app_out * PRICE_OUT
embed_tokens = 0
for usage in EMBED_USAGE:
    embed_tokens += usage["tokens"]
embed_cost = embed_tokens * PRICE_EMBED

per_case = []
for arm, calls in CALLS_PER_CASE.items():
    per_case.append(f"{arm} {calls:.1f}")
p1_per_case = []
for arm, calls in P1_CALLS.items():
    p1_per_case.append(f"{arm} {calls:.1f}")
print(f"judge calls      {judge_calls()}   retried after a 5xx: {len(JUDGE_RETRIES)}   temperature sent: {temperatures}")
print(f"judge calls per case, house rule: {', '.join(per_case)}")
print(f"judge calls per case, P1 rule:    {', '.join(p1_per_case)}")
print(f"judge tokens     {judge_in} in / {judge_out} out   (thinking inside 'out': {judge_thinking}, {100 * judge_thinking / judge_out:.0f}%)")
print(f"judge cost       ${judge_cost:.4f}")
print(f"panel alone      {panel_end - panel_start} calls   ours ${panel_cost:.4f}   DeepEval's counter ${deepeval_cost:.4f}, "
      f"{panel_cost / deepeval_cost:.1f} times less   (input tokens: DeepEval {deepeval_in}, ours {panel_in})")
print(f"app calls        {len(USAGE)}   tokens {app_in} in / {app_out} out   cost ${app_cost:.4f}")
print(f"embed calls      {len(EMBED_USAGE)}   tokens {embed_tokens}   cost ${embed_cost:.4f}")
print(f"total            {judge_calls() + len(USAGE) + len(EMBED_USAGE)} model calls   ${judge_cost + app_cost + embed_cost:.4f}   "
      f"{time.time() - T0:.0f} s since the setup cell")


judge calls      500   retried after a 5xx: 0   temperature sent: [0.0]
judge calls per case, house rule: GEval, criteria 2.0, GEval, our steps 1.0, DAG 3.0
judge calls per case, P1 rule:    GEval, our steps 1.0, GEval, strict 1.0, DAG 3.0
judge tokens     243013 in / 216355 out   (thinking inside 'out': 182751, 84%)
judge cost       $0.9936
panel alone      300 calls   ours $0.6244   DeepEval's counter $0.1912, 3.3 times less   (input tokens: DeepEval 146264, ours 146264)
app calls        19   tokens 5121 in / 10962 out   cost $0.0449
embed calls      12   tokens 2052   cost $0.0003
total            531 model calls   $1.0388   994 s since the setup cell


Five hundred judge calls, none retried, every one at temperature 0.0, the value DeepEval put there. Per case on the house rule: two calls for the paragraph, one for our steps, three for the tree. The same shape on the P1 rule. The judge sent 243,013 tokens in and got 216,355 out. Of those, 182,751 were thinking, 84 percent. The judge cost $\$0.9936$ at the prices we set. For the panel's 300 calls, DeepEval's counter and ours saw the same requests. Ours says $\$0.6244$ and DeepEval's says $\$0.1912$, 3.3 times less. Its input count, 146,264, matched ours to the token. The gap is thinking tokens it does not see, the same shape as the three posts before it. The assistant's 19 calls cost $\$0.0449$ and the 12 embedding calls $\$0.0003$. The whole notebook cost $\$1.0388$ and ran 994 seconds from the setup cell. Most of that time went on the sequential cells after the panel.

## What Halcyon has now, and what it still cannot do

The fingerprint matched, and the question set kept its own fingerprint from the retrieval post. Every score above came from the same pages and tickets as the four posts before it, and the same questions as the retrieval post.

Halcyon now has two house rules written as metrics. The developer-answer rule is a tree. It gates on the deprecation label and the header before it scores anything. It failed the deprecated recipe on every run, where both generator metrics had passed it. The P1 rule exists as three steps a person wrote, and as a two-node tree that prints its path. Both rules are referenceless. Both read only the question, the answer and the chunks, so both could run on the live docs queue tonight. The reference-based metric that saw both traps in the retrieval post cannot. There is no reference for the question a developer is typing now. And the P1 G-Eval the metric-library post left with a moving marking scheme now has a fixed one, at half the judge calls.

What Halcyon still cannot do is say whether the judge is right. Every criterion here was written against twelve pages and ten questions. The key was ours. Nobody has put these answers in front of a Halcyon support engineer and counted where the engineer and the judge disagree. Until someone does, the tree's fifty of fifty is agreement with us, not with Halcyon. A later post on judge calibration takes that up.

The next post is the counterpoint. Most of what a downstream system needs checked needs no judge at all. The severity string, the JSON shape, the header name.

Three rules we are taking forward. We pass the steps in, and a G-Eval with a criterion and no steps does not ship. Any part of a rule that must score zero on its own is a node, and it goes before the judgement of degree. And spread is not accuracy. A metric that agrees with itself on every run has answered the auditor's question, and it has not answered ours.

### Resources

- DeepEval: G-Eval: https://deepeval.com/docs/metrics-llm-evals
- DeepEval: DAG metric: https://deepeval.com/docs/metrics-dag
- DeepEval: building a DAG metric, the guide: https://deepeval.com/guides/guides-dag-metric
- DeepEval: Conversational DAG: https://deepeval.com/docs/metrics-conversational-dag
- G-Eval, the paper: Liu et al., 2023, "G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment": https://arxiv.org/abs/2303.16634
- Latest Gemini models guide, including the removal of sampling parameters: https://ai.google.dev/gemini-api/docs/latest-model
- Gemini API pricing: https://ai.google.dev/gemini-api/docs/pricing
- Companion repository with this notebook and its pyproject.toml: https://github.com/tuhinsharma121/ai-playground/blob/main/eval/deepeval/04-notebook.ipynb